# Data Project
Authors: Alessio Carnevale, Manuel Cattoni, Carlo Schillaci

# Load the Dataset

In [ ]:
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# !pip install duckdb

In [ ]:
from dotenv import load_dotenv
load_dotenv() 

PARQUET = "s3://data-project-supsi-bucket/data.parquet"

In [ ]:
# Configure S3 once
con = duckdb.connect("project.duckdb")
con.sql("INSTALL httpfs; LOAD httpfs;")

# Peek at structure
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{PARQUET}')").df()

In [ ]:
# --- Build papers once, reuse everywhere ---

# Ensure the httpfs extension is loaded (usually implicit, but good to be safe)
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")

# Increase timeout to 5 minutes (300000 milliseconds)
con.sql("SET http_timeout = 300000;")

# Enable retries for temporary network drops
con.sql("SET http_retries = 3;")
con.sql("SET http_retry_wait_ms = 1000;")

REBUILD_PAPERS = False  # set True only when you want to rebuild from S3

if REBUILD_PAPERS:
    con.sql("DROP TABLE IF EXISTS papers")

con.sql(f"""
CREATE TABLE IF NOT EXISTS papers AS
SELECT
    *,
    ARRAY_LENGTH(authors) AS n_authors,
    ARRAY_LENGTH(keywords) AS n_keywords,
    ARRAY_LENGTH("references") AS n_references,
    LENGTH(COALESCE(title, '')) AS title_len,
    LENGTH(COALESCE(abstract, '')) AS abstract_len,
    TRY_CAST(page_end AS BIGINT) - TRY_CAST(page_start AS BIGINT) AS page_count
FROM read_parquet('{PARQUET}')
""")

con.sql("SELECT COUNT(*) AS n_rows FROM papers").df()

In [ ]:
asd = con.sql(f"""
SELECT abstract FROM papers
LIMIT 2
""").df()

asd = np.array(asd)

print(asd[1:][0][0])

In [ ]:
total = con.sql(f"""
    SELECT COUNT(*) AS total_papers
    FROM papers
""").df()


total = np.array(total)

print(f"Total number of papers: {total[0][0]}")

In [ ]:
df_year = con.sql(f"""
    SELECT year, COUNT(*) AS papers
    FROM papers
    WHERE year BETWEEN 1950 AND 2024
    GROUP BY year
    ORDER BY year
""").df()
 
fig, ax = plt.subplots(figsize=(10,4))
ax.fill_between(df_year["year"], df_year["papers"], alpha=0.25, color="steelblue")
ax.plot(df_year["year"], df_year["papers"], color="steelblue", linewidth=2)
ax.set_title("Publications per Year", fontsize=14)
ax.set_xlabel("Year")
ax.set_ylabel("Number of Papers")

plt.tight_layout()
plt.show()
 

In [ ]:
df = con.sql(f"""
    SELECT doc_type, COUNT(*) AS count
    -- FROM papers
    FROM read_parquet('{PARQUET}')
    WHERE doc_type IS NOT NULL
    GROUP BY doc_type ORDER BY count DESC
""").df()

df.plot(kind="bar", x="doc_type", y="count", title="Document Types", figsize=(8,4))
plt.show()

In [ ]:
# Language distribution from parquet
df = con.sql(f"""
    SELECT lang, COUNT(*) AS count
    -- FROM papers
    FROM read_parquet('{PARQUET}')
    WHERE lang IS NOT NULL
    GROUP BY lang 
    ORDER BY count DESC
    LIMIT 15
""").df()

df.plot(kind="bar", x="lang", y="count", title="Language Distribution", figsize=(10, 5))
plt.tight_layout()
plt.show()

## Citation Analysis

In [ ]:
df = con.sql(f"""
    SELECT n_citation
    -- FROM papers
    FROM read_parquet('{PARQUET}')
    WHERE n_citation IS NOT NULL AND n_citation > 0
    USING SAMPLE 20000
""").df()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df["n_citation"].plot(kind="hist", bins=50, ax=axes[0], title="Citations (linear scale)")
df["n_citation"].apply(lambda x: x+1).plot(kind="hist", bins=50, logy=True, ax=axes[1], title="Citations (log scale)")
plt.tight_layout()
plt.show()

In [ ]:
df = con.sql(f"""
    SELECT doc_type,
           COUNT(*) AS papers,
           AVG(n_citation) AS avg_citations,
           MEDIAN(n_citation) AS median_citations
    
    -- FROM papers         
    FROM read_parquet('{PARQUET}')
    WHERE doc_type IS NOT NULL AND n_citation IS NOT NULL
    GROUP BY doc_type ORDER BY median_citations DESC
""").df()

df.plot(kind="bar", x="doc_type", y="median_citations", title="Median Citations by Doc Type", figsize=(8,4))
plt.show()

In [ ]:
df = con.sql(f"""
    SELECT year,
           MEDIAN(n_citation) AS median_citations,
           COUNT(*) AS papers
    -- FROM papers
    FROM read_parquet('{PARQUET}')
    WHERE year BETWEEN 1980 AND 2020 AND n_citation IS NOT NULL
    GROUP BY year ORDER BY year
""").df()

fig, ax1 = plt.subplots(figsize=(12, 4))
ax2 = ax1.twinx()
ax1.plot(df["year"], df["median_citations"], color="blue", label="Median citations")
ax2.bar(df["year"], df["papers"], alpha=0.3, color="gray", label="Paper count")
ax1.set_title("Citations & Volume Over Time")
plt.show()

## Authors and Collaborations

In [ ]:
df = con.sql(f"""
    SELECT year,
           AVG(ARRAY_LENGTH(authors)) AS avg_authors
 -- FROM papers
    FROM read_parquet('{PARQUET}')
    WHERE year BETWEEN 1980 AND 2024 AND authors IS NOT NULL
    GROUP BY year ORDER BY year
""").df()

df.plot(kind="line", x="year", y="avg_authors", title="Avg Authors per Paper Over Time", figsize=(12,4))
plt.show()

In [ ]:
df = con.sql(f"""
    SELECT
        author.name AS author_name,
        COUNT(*) AS papers
    -- FROM papers
    FROM read_parquet('{PARQUET}')
    CROSS JOIN UNNEST(authors) AS t(author)
    WHERE author.name IS NOT NULL
      AND author.name != ''
    GROUP BY 1
    ORDER BY papers DESC
    LIMIT 20
""").df()

df.plot(kind="barh", x="author_name", y="papers", title="Top 20 Most Prolific Authors", figsize=(8,8))
plt.gca().invert_yaxis()
plt.show()

## Keywords and Venues

In [ ]:
df = con.sql(f"""
    SELECT kw AS keyword, COUNT(*) AS count
    -- FROM papers
    FROM read_parquet('{PARQUET}')
    CROSS JOIN UNNEST(keywords) AS t(kw)
    WHERE kw IS NOT NULL AND kw != ''
    GROUP BY kw
    ORDER BY count DESC
    LIMIT 30
""").df()

df.plot(kind="barh", x="keyword", y="count", title="Top 30 Keywords", figsize=(8,10))
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# merge identical keywords by case

def to_camel_case(s):
    return ' '.join(word.capitalize() for word in s.split())

raw = con.sql(f"""
    SELECT kw AS keyword
    FROM read_parquet('{PARQUET}')
    -- FROM papers
    CROSS JOIN
         UNNEST(keywords) AS t(kw)
    WHERE kw IS NOT NULL AND kw != ''
""").df()

raw['keyword'] = raw['keyword'].apply(to_camel_case)

df = (
    raw.groupby('keyword', as_index=False)
       .size()
       .rename(columns={'size': 'count'})
       .sort_values('count', ascending=False)
       .head(30)
)

df.plot(kind="barh", x="keyword", y="count", title="Top 30 Keywords", figsize=(8, 10))
plt.gca().invert_yaxis()
plt.show()

In [ ]:
df = con.sql(f"""
    SELECT venue, COUNT(*) AS papers, MEDIAN(n_citation) AS median_citations
    -- FROM papers
    FROM read_parquet('{PARQUET}')
    WHERE venue IS NOT NULL AND venue != ''
    GROUP BY venue
    ORDER BY papers DESC
    LIMIT 20
""").df()

df.plot(kind="barh", x="venue", y="papers", title="Top 20 Venues by Volume", figsize=(8,8))
plt.gca().invert_yaxis()
plt.show()

##  Missingness

In [ ]:
# Missingness report (null + empty string/array) from parquet
missing_df = con.sql(f"""
WITH base AS (
    SELECT
        COUNT(*)::DOUBLE AS total_rows,
        SUM(CASE WHEN id IS NULL OR TRIM(COALESCE(id, '')) = '' THEN 1 ELSE 0 END) AS miss_id,
        SUM(CASE WHEN title IS NULL OR TRIM(COALESCE(title, '')) = '' THEN 1 ELSE 0 END) AS miss_title,
        SUM(CASE WHEN abstract IS NULL OR TRIM(COALESCE(abstract, '')) = '' THEN 1 ELSE 0 END) AS miss_abstract,
        SUM(CASE WHEN keywords IS NULL OR ARRAY_LENGTH(keywords) = 0 THEN 1 ELSE 0 END) AS miss_keywords,
        SUM(CASE WHEN year IS NULL THEN 1 ELSE 0 END) AS miss_year,
        SUM(CASE WHEN authors IS NULL OR ARRAY_LENGTH(authors) = 0 THEN 1 ELSE 0 END) AS miss_authors,
        SUM(CASE WHEN "references" IS NULL OR ARRAY_LENGTH("references") = 0 THEN 1 ELSE 0 END) AS miss_references,
        SUM(CASE WHEN lang IS NULL OR TRIM(COALESCE(lang, '')) = '' THEN 1 ELSE 0 END) AS miss_lang,
        SUM(CASE WHEN venue IS NULL OR TRIM(COALESCE(venue, '')) = '' THEN 1 ELSE 0 END) AS miss_venue,
        SUM(CASE WHEN doc_type IS NULL OR TRIM(COALESCE(doc_type, '')) = '' THEN 1 ELSE 0 END) AS miss_doc_type,
        SUM(CASE WHEN doi IS NULL OR TRIM(COALESCE(doi, '')) = '' THEN 1 ELSE 0 END) AS miss_doi,
        SUM(CASE WHEN page_start IS NULL OR TRIM(COALESCE(page_start, '')) = '' THEN 1 ELSE 0 END) AS miss_page_start,
        SUM(CASE WHEN page_end IS NULL OR TRIM(COALESCE(page_end, '')) = '' THEN 1 ELSE 0 END) AS miss_page_end,
        SUM(CASE WHEN n_citation IS NULL THEN 1 ELSE 0 END) AS miss_n_citation
    FROM read_parquet('{PARQUET}')
    -- FROM papers
)
SELECT * FROM (
    SELECT 'id' AS feature,        100.0 * miss_id / total_rows AS missing_pct FROM base
    UNION ALL SELECT 'title',      100.0 * miss_title / total_rows FROM base
    UNION ALL SELECT 'abstract',   100.0 * miss_abstract / total_rows FROM base
    UNION ALL SELECT 'keywords',   100.0 * miss_keywords / total_rows FROM base
    UNION ALL SELECT 'year',       100.0 * miss_year / total_rows FROM base
    UNION ALL SELECT 'authors',    100.0 * miss_authors / total_rows FROM base
    UNION ALL SELECT 'references', 100.0 * miss_references / total_rows FROM base
    UNION ALL SELECT 'lang',       100.0 * miss_lang / total_rows FROM base
    UNION ALL SELECT 'venue',      100.0 * miss_venue / total_rows FROM base
    UNION ALL SELECT 'doc_type',   100.0 * miss_doc_type / total_rows FROM base
    UNION ALL SELECT 'doi',        100.0 * miss_doi / total_rows FROM base
    UNION ALL SELECT 'page_start', 100.0 * miss_page_start / total_rows FROM base
    UNION ALL SELECT 'page_end',   100.0 * miss_page_end / total_rows FROM base
    UNION ALL SELECT 'n_citation', 100.0 * miss_n_citation / total_rows FROM base
)
ORDER BY missing_pct DESC
""").df()

display(missing_df)

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=missing_df, x="feature", y="missing_pct", color="steelblue")
plt.title("Missingness % by Feature")
plt.ylabel("Missing %")
plt.xlabel("")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Data Quality

In [ ]:
df = con.sql(f"""
    SELECT
        COUNT(*) AS total,
        COUNT(title) AS has_title,
        COUNT(abstract) AS has_abstract,
        COUNT(year) AS has_year,
        COUNT(doi) AS has_doi,
        COUNT(n_citation) AS has_citations,
        COUNT(venue) AS has_venue
    -- FROM papers
    FROM read_parquet('{PARQUET}')
""").df()

print(df.T)

# Data quality

In [ ]:
df = con.sql(f"""
    SELECT
        COUNT(*) AS total,
        COUNT(title) AS has_title,
        COUNT(abstract) AS has_abstract,
        COUNT(year) AS has_year,
        COUNT(doi) AS has_doi,
        COUNT(n_citation) AS has_citations,
        COUNT(venue) AS has_venue
    -- FROM papers
    FROM read_parquet('{PARQUET}')
""").df()

print(df.T)

## What this next check does (author.id / author.org missingness)
It explodes the authors array so each author mention in each paper is one row.
Then it counts how often author.id is missing, how often author.org is missing, and how often both are missing.
Output is a table with counts and percentages.
How to read it:

- High miss_author_id_pct means many authors cannot be uniquely tracked across papers.
- High miss_author_org_pct means affiliation-based features may be weak/noisy.
- High miss_both_pct means severe metadata incompleteness for author-level modeling.

In [ ]:
# Missing author.id and author.org rates (explicit)
missing_author_fields_df = con.sql(f"""
WITH a AS (
    SELECT
        author.name AS author_name,
        author.id   AS author_id,
        author.org  AS author_org
    -- FROM papers
    FROM read_parquet('{PARQUET}')
    CROSS JOIN UNNEST(authors) AS t(author)
),
base AS (
    SELECT
        COUNT(*)::DOUBLE AS total_author_mentions,
        SUM(CASE WHEN author_id IS NULL OR TRIM(COALESCE(author_id, '')) = '' THEN 1 ELSE 0 END) AS miss_author_id,
        SUM(CASE WHEN author_org IS NULL OR TRIM(COALESCE(author_org, '')) = '' THEN 1 ELSE 0 END) AS miss_author_org,
        SUM(CASE WHEN (author_id IS NULL OR TRIM(COALESCE(author_id, '')) = '')
                  AND (author_org IS NULL OR TRIM(COALESCE(author_org, '')) = '')
                 THEN 1 ELSE 0 END) AS miss_both
    FROM a
)
SELECT
    total_author_mentions,
    miss_author_id,
    ROUND(100.0 * miss_author_id / NULLIF(total_author_mentions, 0), 4) AS miss_author_id_pct,
    miss_author_org,
    ROUND(100.0 * miss_author_org / NULLIF(total_author_mentions, 0), 4) AS miss_author_org_pct,
    miss_both,
    ROUND(100.0 * miss_both / NULLIF(total_author_mentions, 0), 4) AS miss_both_pct
FROM base;
""").df()
display(missing_author_fields_df)

author.id missing ≈ 9.94% author.org missing ≈ 18.30% both missing ≈ 3.48%

Missingness is too high for author.org; dropping would bias data

We should create missingness features. booleans like: has_author_id, has_author_org, and pair-level both_have_org.

Use author.id as primary identity, normalized name only as fallback.

With this class rate, train on sampled negatives (e.g., 1:5 or 1:10 pos:neg) and evaluate with PR-AUC / Recall@K, not accuracy.

## What this next check does (author-name variation / normalization)
It normalizes names (lowercase, remove punctuation, collapse spaces).
It groups by normalized form and finds cases where many raw spellings map to the same normalized author string.
Output shows likely spelling/format variants (for example accents, dots, extra spaces, initials).
How to read it:

- n_raw_variants = number of distinct original spellings for the same normalized name.
- mentions = total occurrences of that normalized name.
- raw_variants lists the observed spellings with counts.
Important:

This is a heuristic quality check, not true entity resolution.
Same normalized name can still refer to different people (homonyms).

In [ ]:
# 2) Author-name variation / normalization checks (memory-safe)

# DuckDB tuning for this heavy aggregation
con.sql("SET preserve_insertion_order = false;")
con.sql("SET threads = 4;")

name_variation_df = con.sql(f"""
WITH flat AS (
    SELECT TRIM(author.name) AS raw_name
    -- FROM papers
    FROM read_parquet('{PARQUET}')
    CROSS JOIN UNNEST(authors) AS t(author)
    WHERE author.name IS NOT NULL AND TRIM(author.name) <> ''
),
norm_counts AS (
    SELECT
        LOWER(
            REGEXP_REPLACE(
                REGEXP_REPLACE(raw_name, '[^[:alnum:] ]', ' ', 'g'),
                '\\s+', ' ', 'g'
            )
        ) AS normalized_name,
        raw_name,
        COUNT(*) AS raw_mentions
    FROM flat
    GROUP BY 1, 2
),
name_stats AS (
    SELECT
        normalized_name,
        SUM(raw_mentions) AS mentions,
        COUNT(*) AS n_raw_variants
    FROM norm_counts
    GROUP BY normalized_name
    HAVING COUNT(*) > 1
    ORDER BY n_raw_variants DESC, mentions DESC
    LIMIT 30
)
SELECT
    s.normalized_name,
    s.mentions,
    s.n_raw_variants,
    STRING_AGG(
        nc.raw_name || ' (' || CAST(nc.raw_mentions AS VARCHAR) || ')',
        ' | '
        ORDER BY nc.raw_mentions DESC, nc.raw_name
    ) AS raw_variants
FROM name_stats s
JOIN norm_counts nc USING (normalized_name)
GROUP BY s.normalized_name, s.mentions, s.n_raw_variants
ORDER BY s.n_raw_variants DESC, s.mentions DESC;
""").df()

display(name_variation_df)

The name-variation table is for finding likely spelling variants; treat it as cleaning support, not truth.

We should build canonical text forms (lowercase, trim, punctuation/space cleanup, unicode normalization). Keep a small manual alias map for top recurring variants from the variation output

## What this next check does (temporal consistency of affiliations)
- It keeps authors with a valid author.id and year.
- For each author across time, it counts how many distinct non-empty organizations appear.
- It returns authors with more than one organization over their timeline.

How to read it:

- n_distinct_orgs > 1 can mean real career moves or inconsistent/dirty org strings.
- timeline_rows and year range (first_year, last_year) give context on how broad the timeline is.
- Rows at the top are the strongest candidates for manual cleaning rules (org normalization, alias mapping).

In [ ]:
# Temporal consistency of author affiliations
author_org_temporal_df = con.sql(f"""
WITH author_timeline AS (
    SELECT
        author.id AS author_id,
        MIN(TRIM(author.name)) AS sample_name,
        year,
        NULLIF(TRIM(author.org), '') AS org
    -- FROM papers
    FROM read_parquet('{PARQUET}')
    CROSS JOIN UNNEST(authors) AS t(author)
    WHERE author.id IS NOT NULL AND TRIM(author.id) <> ''
      AND year IS NOT NULL
    GROUP BY author.id, year, NULLIF(TRIM(author.org), '')
)
 ,agg AS (
    SELECT
        author_id,
        MIN(sample_name) AS sample_name,
        COUNT(*) AS timeline_rows,
        COUNT(DISTINCT org) FILTER (WHERE org IS NOT NULL) AS n_distinct_orgs,
        MIN(year) AS first_year,
        MAX(year) AS last_year
    FROM author_timeline
    GROUP BY author_id
)
SELECT
    author_id,
    sample_name,
    timeline_rows,
    n_distinct_orgs,
    first_year,
    last_year
FROM agg
WHERE n_distinct_orgs > 1
ORDER BY n_distinct_orgs DESC, timeline_rows DESC
LIMIT 30;
""").df()

display(author_org_temporal_df)

for example, the first row shows that:

The same author ID appears a lot in the dataset (786 author-year-org records). Across those records, there are 510 different organization strings. This is far too high to be realistic as true job changes, so it mostly indicates: noisy/inconsistent author.org text, variants of the same institution, possible mixed identities in the source.

What to do with it:

Keep author.id as the stable identity key. Normalize author.org (case, punctuation, spacing, abbreviations). Build an org alias map for top repeated variants. For modeling, use robust org features (has_org, normalized org match) rather than raw org string equality.

# Data Cleaning

In [ ]:
import re
import unicodedata

def normalize_text(value):
    if value is None:
        return ""
    text = unicodedata.normalize("NFKD", str(value))
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

try:
    con.create_function("normalize_text", normalize_text, [str], str)
except Exception:
    pass

Author canonicalization layer: 'author.id' is the primary key, and normalized name is the fallback.

In [ ]:
con.sql(f"""
CREATE OR REPLACE TABLE author_mentions_clean AS
SELECT
    p.id AS paper_id,
    p.year,
    CASE
        WHEN a.author.id IS NOT NULL AND TRIM(a.author.id) <> ''
            THEN TRIM(a.author.id)
        ELSE 'name:' || normalize_text(a.author.name)
    END AS author_key,
    TRIM(a.author.id) AS author_id_raw,
    normalize_text(a.author.name) AS author_name_norm,
    TRIM(a.author.name) AS author_name_raw,
    normalize_text(a.author.org) AS author_org_norm
FROM read_parquet('{PARQUET}') p
-- FROM papers p
CROSS JOIN UNNEST(p.authors) AS a(author)
WHERE a.author.name IS NOT NULL
  AND TRIM(a.author.name) <> '';
""")

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE author_name_canonical AS
WITH counts AS (
    SELECT
        author_key,
        author_name_norm,
        author_name_raw,
        COUNT(*) AS n_mentions
    FROM author_mentions_clean
    GROUP BY 1, 2, 3
),
ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY author_key
            ORDER BY n_mentions DESC, LENGTH(author_name_raw) DESC, author_name_raw
        ) AS rn
    FROM counts
)
SELECT
    author_key,
    author_name_raw AS canonical_author_name,
    author_name_norm AS canonical_author_name_norm,
    n_mentions
FROM ranked
WHERE rn = 1;
""")

In [ ]:
display(con.sql("SELECT * FROM author_name_canonical LIMIT 10").df())

Use the resolved author-table cells below. The old raw-name join should not be used.

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE paper_authors AS
SELECT
    am.paper_id AS id,
    am.author_key,
    ac.canonical_author_name AS author_name,
    am.author_org_norm
FROM author_mentions_clean am
LEFT JOIN author_name_canonical ac
    ON am.author_key = ac.author_key;
""")

con.sql("""
CREATE OR REPLACE TABLE author_stats AS
SELECT
    author_key,
    author_name,
    COUNT(*) AS author_papers,
    MEDIAN(p.n_citation) AS author_median_citations,
    AVG(p.n_citation) AS author_avg_citations,
    MAX(p.n_citation) AS author_max_citations
FROM paper_authors pa
JOIN papers p
    ON pa.id = p.id
GROUP BY author_key, author_name;
""")

con.sql(f"""
CREATE OR REPLACE TABLE features_author AS
SELECT
    p.id,
    COUNT(DISTINCT pa.author_key) AS n_authors,
    COALESCE(AVG(a.author_avg_citations), 0) AS avg_author_citations,
    COALESCE(MAX(a.author_max_citations), 0) AS max_author_citations,
    CASE WHEN COUNT(DISTINCT pa.author_key) > 1 THEN 1 ELSE 0 END AS multi_authored
FROM read_parquet('{PARQUET}') p
-- FROM papers p
LEFT JOIN paper_authors pa
    ON p.id = pa.id
LEFT JOIN author_stats a
    ON pa.author_key = a.author_key
GROUP BY p.id;
""")

display(con.sql("SELECT * FROM paper_authors LIMIT 10").df())

In [ ]:
con.sql(f"""
CREATE OR REPLACE TABLE papers_clean AS
WITH base AS (
    SELECT
        p.*,
        normalize_text(p.title) AS title_norm,
        normalize_text(p.abstract) AS abstract_norm,
        normalize_text(p.venue) AS venue_norm,
        normalize_text(p.doc_type) AS doc_type_norm,
        CASE
            WHEN p.id IS NULL OR TRIM(p.id) = '' THEN
                'paper_' || md5(
                    COALESCE(p.title, '') || '|' ||
                    COALESCE(p.abstract, '') || '|' ||
                    COALESCE(CAST(p.year AS VARCHAR), '') || '|' ||
                    COALESCE(p.venue, '')
                )
            ELSE TRIM(p.id)
        END AS id_clean,
        CASE WHEN p.id IS NULL OR TRIM(p.id) = '' THEN 1 ELSE 0 END AS was_missing_id
    -- FROM papers p
    FROM read_parquet('{PARQUET}') p
)
SELECT
    id_clean AS id,
    NULLIF(title_norm, '') AS title,
    NULLIF(abstract_norm, '') AS abstract,
    year,
    authors,
    keywords,
    "references",
    NULLIF(venue_norm, '') AS venue,
    CASE
        WHEN regexp_matches(COALESCE(venue_norm, ''), '(conference|symposium|workshop|proceedings|meeting)') THEN 'conference'
        WHEN regexp_matches(COALESCE(venue_norm, ''), '(journal|transactions|letters)') THEN 'journal'
        WHEN regexp_matches(COALESCE(venue_norm, ''), '(book|chapter|lecture notes)') THEN 'book'
        WHEN doc_type_norm IN ('article', 'journal article') THEN 'journal'
        WHEN doc_type_norm IN ('inproceedings', 'conference paper', 'conference') THEN 'conference'
        WHEN doc_type_norm = '' THEN 'unknown'
        ELSE doc_type_norm
    END AS doc_type,
    doi,
    page_start,
    page_end,
    n_citation,
    ARRAY_LENGTH(authors) AS n_authors,
    ARRAY_LENGTH(keywords) AS n_keywords,
    ARRAY_LENGTH("references") AS n_references,
    LENGTH(COALESCE(NULLIF(title_norm, ''), '')) AS title_len,
    LENGTH(COALESCE(NULLIF(abstract_norm, ''), '')) AS abstract_len,
    TRY_CAST(page_end AS BIGINT) - TRY_CAST(page_start AS BIGINT) AS page_count,
    was_missing_id
FROM base;
""")

con.sql("CREATE OR REPLACE TABLE papers AS SELECT * FROM papers_clean")

display(con.sql("SELECT id, title, venue, doc_type, was_missing_id FROM papers LIMIT 5").df())

# Target Focused Exploration

In [ ]:
# Class balance: citation links (positive) vs non-links (negative)

# Assumptions:
## - `papers.id` is the paper identifier
## - `papers.references` is an array of cited paper ids
## - we only keep references that point to ids present in `papers`

# Set temp directory for DuckDB (Windows fix)
import tempfile
temp_dir = tempfile.gettempdir()
con.sql(f"SET temp_directory = '{temp_dir.replace(chr(92), '/')}'")

balance_df = con.sql(f"""
WITH ids AS (
    SELECT DISTINCT id
    -- FROM papers
    FROM read_parquet('{PARQUET}')
    WHERE id IS NOT NULL AND TRIM(id) <> ''
)
 ,pos AS (
    SELECT DISTINCT
        p.id AS src_id,
        ref AS dst_id
    -- FROM papers p
    FROM read_parquet('{PARQUET}') p
    CROSS JOIN UNNEST(p."references") AS t(ref)
    INNER JOIN ids i ON i.id = ref
    WHERE p.id IS NOT NULL
      AND TRIM(p.id) <> ''
      AND ref IS NOT NULL
      AND TRIM(ref) <> ''
      AND p.id <> ref
)
 ,counts AS (
    SELECT
        (SELECT COUNT(*) FROM ids) AS n_nodes,
        (SELECT COUNT(*) FROM pos) AS n_positive_links
)
SELECT
    n_nodes,
    n_positive_links,
    (n_nodes * (n_nodes - 1) - n_positive_links) AS n_non_links,
    ROUND(100.0 * n_positive_links / NULLIF(n_nodes * (n_nodes - 1), 0), 6) AS positive_rate_pct,
    ROUND(100.0 * (n_nodes * (n_nodes - 1) - n_positive_links) / NULLIF(n_nodes * (n_nodes - 1), 0), 6) AS non_link_rate_pct
FROM counts;
""").df()

display(balance_df)

In [ ]:
# ratio as "1 positive : X negatives"
pos = float(balance_df.loc[0, "n_positive_links"])
neg = float(balance_df.loc[0, "n_non_links"])
print(f"Positive link rate: {balance_df.loc[0, 'positive_rate_pct']:.6f}%")
print(f"Non-link rate: {balance_df.loc[0, 'non_link_rate_pct']:.6f}%")

# Pair Level Analysis

## Features selection

### Correlation Matrix

In [ ]:
# df = con.sql(f"""
#     SELECT
#         ARRAY_LENGTH(authors) AS n_authors,
#         ARRAY_LENGTH(keywords) AS n_keywords,
#         ARRAY_LENGTH("references") AS n_references,
#         TRY_CAST(page_end AS INT) - TRY_CAST(page_start AS INT) AS page_count,
#         LENGTH(abstract) AS abstract_length,
#         n_citation
#     FROM read_parquet('{PARQUET}')
#     WHERE n_citation IS NOT NULL
#     USING SAMPLE 20000
# """).df().dropna()

# sns.heatmap(df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
# plt.title("Feature Correlations")
# plt.tight_layout()
# plt.show()

There are no correlated features in the original dataset, only a little correlation between n_references and abstract_lenght.

# Feature Selection

### 1 TEXT-BASED FEATURES

Features da **titolo** e **abstract** per catturare complessità e specificità:
- `title_word_count` → Lunghezza titolo
- `abstract_word_count` → Lunghezza abstract  
- `title_complexity` → Media parole nel titolo (linguaggio tecnico)
- `abstract_complexity` → Media parole nell'abstract
- `has_numbers_*` → Presenza numeri (specificità)

In [ ]:
text_sql = """
CREATE OR REPLACE TABLE features_text AS
SELECT 
    id,
    CASE WHEN title IS NULL THEN 0 
         ELSE ARRAY_LENGTH(STRING_SPLIT(TRIM(title), ' '))
    END AS title_word_count,
    
    CASE WHEN abstract IS NULL THEN 0 
         ELSE ARRAY_LENGTH(STRING_SPLIT(TRIM(abstract), ' '))
    END AS abstract_word_count,
    
    CASE WHEN LENGTH(COALESCE(abstract, '')) = 0 THEN 0
         ELSE CAST(LENGTH(COALESCE(title, '')) AS FLOAT) / LENGTH(COALESCE(abstract, ''))
    END AS title_abstract_ratio,
    
    CASE WHEN title IS NULL OR ARRAY_LENGTH(STRING_SPLIT(TRIM(title), ' ')) = 0 THEN 0
         ELSE CAST(LENGTH(COALESCE(title, '')) AS FLOAT) / ARRAY_LENGTH(STRING_SPLIT(TRIM(title), ' '))
    END AS title_complexity,
    
    CASE WHEN abstract IS NULL OR ARRAY_LENGTH(STRING_SPLIT(TRIM(abstract), ' ')) = 0 THEN 0
         ELSE CAST(LENGTH(COALESCE(abstract, '')) AS FLOAT) / ARRAY_LENGTH(STRING_SPLIT(TRIM(abstract), ' '))
    END AS abstract_complexity,
    
    CASE WHEN title ~ '[0-9]' THEN 1 ELSE 0 END AS has_numbers_in_title,
    CASE WHEN abstract ~ '[0-9]' THEN 1 ELSE 0 END AS has_numbers_in_abstract
    
FROM read_parquet('{PARQUET}')
""".format(PARQUET=PARQUET)

con.sql(text_sql)
print("DONE")

### 2️ AUTHOR-BASED FEATURES

Features che catturano **reputazione** e **collaborazione**:
- `n_authors` → Numero autori  
- `first_author_avg_citations` → Media cit. primo autore (star lead researcher)
- `avg_author_citations` → Media cit. aggregato autori
- `max_author_citations` → Max cit. tra autori (presenza di star)
- `multi_authored` → Flag collaborazione (>1 autore)

In [ ]:
# Compute author statistics
author_stats_sql = """
CREATE OR REPLACE TABLE author_stats AS
SELECT 
    author.name AS author_name,
    COUNT(*) AS author_papers,
    MEDIAN(n_citation) AS author_median_citations,
    AVG(n_citation) AS author_avg_citations,
    MAX(n_citation) AS author_max_citations
FROM read_parquet('{PARQUET}') 
CROSS JOIN UNNEST(authors) AS t(author)
WHERE author.name IS NOT NULL AND author.name != ''
GROUP BY author.name
""".format(PARQUET=PARQUET)

con.sql(author_stats_sql)
print("Author statistics DONE")

# Create paper-author links
paper_author_sql = """
CREATE OR REPLACE TABLE paper_authors AS
SELECT 
    p.id,
    author.name AS author_name
FROM read_parquet('{PARQUET}') AS p
CROSS JOIN UNNEST(p.authors) AS t(author)
WHERE author.name IS NOT NULL AND author.name != ''
""".format(PARQUET=PARQUET)

con.sql(paper_author_sql)
print("Paper-author links DONE")

# Join with author stats
author_features_sql = """
CREATE OR REPLACE TABLE features_author AS
SELECT 
    p.id,
    ARRAY_LENGTH(p.authors) AS n_authors,
    
    COALESCE(AVG(a.author_avg_citations), 0) AS avg_author_citations,
    COALESCE(MAX(a.author_max_citations), 0) AS max_author_citations,
    
    CASE WHEN ARRAY_LENGTH(p.authors) > 1 THEN 1 ELSE 0 END AS multi_authored

FROM read_parquet('{PARQUET}') AS p
LEFT JOIN paper_authors pa ON p.id = pa.id
LEFT JOIN author_stats a ON pa.author_name = a.author_name
GROUP BY p.id, ARRAY_LENGTH(p.authors)
""".format(PARQUET=PARQUET)

con.sql(author_features_sql)
print("AUTHOR-BASED FEATURES DONE")

### 3 NETWORK & REFERENCE FEATURES

- `n_references` → Numero di riferimenti citati  
- `references_author_ratio` → Riferimenti per autore
- `citation_richness` → Densità riferimenti rispetto abstract
- `has_high_ref_count` → Flag papers che sono review/comprehensive (>50 ref)

In [ ]:
network_sql = """
CREATE OR REPLACE TABLE features_network AS
SELECT 
    id,
    ARRAY_LENGTH("references") AS n_references,
    
    CASE WHEN ARRAY_LENGTH(authors) = 0 THEN 0
         ELSE CAST(ARRAY_LENGTH("references") AS FLOAT) / ARRAY_LENGTH(authors)
    END AS references_author_ratio,
    
    CASE WHEN LENGTH(abstract) = 0 THEN 0
         ELSE CAST(ARRAY_LENGTH("references") AS FLOAT) / (LENGTH(abstract) / 100.0)
    END AS citation_richness,
    
    CASE WHEN ARRAY_LENGTH("references") > 50 THEN 1 ELSE 0 END AS has_high_ref_count

FROM read_parquet('{PARQUET}')
""".format(PARQUET=PARQUET)

con.sql(network_sql)
print("NETWORK & REFERENCE FEATURES DONE")

### 4 TEMPORAL FEATURES

Features che catturano **recency** e **stagionalità**:
- `years_since_publication` → Anni dalla pubblicazione (non usare lineare!)
- `publication_era` → Era storica (pre-2000, 2000-2009, 2010-2019, 2020+)
- `decade` → Decennio della pubblicazione
- `is_recent` → Flag se pubblicato negli ultimi 5 anni

In [ ]:
temporal_sql = """
CREATE OR REPLACE TABLE features_temporal AS
SELECT 
    id,
    year,
    (2024 - year) AS years_since_publication,
    
    CASE 
        WHEN year < 2000 THEN 'pre-2000'
        WHEN year < 2010 THEN '2000-2009'
        WHEN year < 2020 THEN '2010-2019'
        ELSE '2020+'
    END AS publication_era,
    
    CAST(year / 10 * 10 AS INT) AS decade,
    
    CASE WHEN year >= 2019 THEN 1 ELSE 0 END AS is_recent

FROM read_parquet('{PARQUET}')
WHERE year IS NOT NULL
""".format(PARQUET=PARQUET)

con.sql(temporal_sql)
print("TEMPORAL FEATURES DONE")

### 5 QUALITY & IMPACT FEATURES

Features che rappresentano **qualità e impatto** normalizzati nel tempo:
- `citation_per_year` → Velocità accumulo citazioni (normalizzato per recency)  
- `is_highly_cited_for_year` → Flag se sopra mediana dell'anno
- `impact_score` → Composito: (cit/years × ln(peers per year)) **BEST predictor**

In [ ]:
# Compute year statistics for normalization
con.sql("""
CREATE OR REPLACE TABLE year_citation_stats AS
SELECT 
    year,
    MEDIAN(n_citation) AS median_citations_per_year,
    AVG(n_citation) AS avg_citations_per_year,
    COUNT(*) AS papers_per_year
FROM read_parquet('{PARQUET}')
WHERE year IS NOT NULL AND n_citation IS NOT NULL
GROUP BY year
""".format(PARQUET=PARQUET))

quality_sql = """
CREATE OR REPLACE TABLE features_quality AS
SELECT 
    p.id,
    p.year,
    p.n_citation,
    
    CASE 
        WHEN (2024 - p.year) = 0 THEN CAST(p.n_citation AS FLOAT)
        WHEN p.n_citation = 0 THEN 0.0
        ELSE CAST(p.n_citation AS FLOAT) / CAST((2024 - p.year) AS FLOAT)
    END AS citation_per_year,
    
    CASE 
        WHEN p.n_citation >= COALESCE(ycs.median_citations_per_year, 0) THEN 1 
        ELSE 0 
    END AS is_highly_cited_for_year,
    
    (CAST(p.n_citation AS FLOAT) / CAST((2024 - p.year + 1) AS FLOAT)) * 
    (1 + LN(CAST(COALESCE(ycs.papers_per_year, 1) AS FLOAT))) AS impact_score
    
FROM read_parquet('{PARQUET}') AS p
LEFT JOIN year_citation_stats AS ycs ON p.year = ycs.year
WHERE p.n_citation IS NOT NULL
""".format(PARQUET=PARQUET)

con.sql(quality_sql)
print("QUALITY & IMPACT FEATURES DONE")

### 6 VENUE & DOMAIN FEATURES

Features che catturano **prestigio del venue** e **accessibility**:
- `venue_median_citations` → Prestigi proxy del venue (STRONGEST PREDICTOR)
- `venue_volume` → Numero papers nel venue (importance marker)
- `is_top_venue` → Flag se venue è nei top 50
- `is_english` → Flag lingua inglese (global reach)
- `venue_tier` → Ordinale categorico (elite → high → medium → low)

In [ ]:
# Compute venue statistics
venue_stats_sql = """
CREATE OR REPLACE TABLE venue_stats AS
SELECT 
    venue,
    COUNT(*) AS venue_papers,
    MEDIAN(n_citation) AS venue_median_citations,
    AVG(n_citation) AS venue_avg_citations
FROM read_parquet('{PARQUET}')
WHERE venue IS NOT NULL AND venue != '' AND n_citation IS NOT NULL
GROUP BY venue
""".format(PARQUET=PARQUET)

con.sql(venue_stats_sql)

# Get top 50 venues
con.sql("CREATE OR REPLACE TABLE top_venues AS SELECT venue FROM venue_stats ORDER BY venue_median_citations DESC LIMIT 50")

# Create venue features
venue_features_sql = """
CREATE OR REPLACE TABLE features_venue AS
SELECT 
    p.id,
    p.venue,
    p.lang,
    
    COALESCE(vs.venue_median_citations, 0) AS venue_median_citations,
    COALESCE(vs.venue_papers, 0) AS venue_volume,
    
    CASE WHEN p.venue IN (SELECT venue FROM top_venues) THEN 1 ELSE 0 END AS is_top_venue,
    
    CASE WHEN UPPER(p.lang) IN ('EN', 'ENGLISH') THEN 1 ELSE 0 END AS is_english,
    
    CASE 
        WHEN COALESCE(vs.venue_median_citations, 0) > 100 THEN 'elite'
        WHEN COALESCE(vs.venue_median_citations, 0) > 50 THEN 'high'
        WHEN COALESCE(vs.venue_median_citations, 0) > 10 THEN 'medium'
        ELSE 'low'
    END AS venue_tier

FROM read_parquet('{PARQUET}') AS p
LEFT JOIN venue_stats AS vs ON p.venue = vs.venue
""".format(PARQUET=PARQUET)

con.sql(venue_features_sql)
print("VENUE & DOMAIN FEATURES DONE")

###  CONSOLIDATE: Master Features Table

In [ ]:
master_sql = """
CREATE OR REPLACE TABLE papers_with_features AS
SELECT 
    p.id,
    p.title,
    p.year,
    p.n_citation AS target_variable,
    
    -- Text features
    ft.title_word_count,
    ft.abstract_word_count,
    ft.title_abstract_ratio,
    ft.title_complexity,
    ft.abstract_complexity,
    ft.has_numbers_in_title,
    ft.has_numbers_in_abstract,
    
    -- Author features
    fa.n_authors,
    fa.avg_author_citations,
    fa.max_author_citations,
    fa.multi_authored,
    
    -- Network features
    fn.n_references,
    fn.references_author_ratio,
    fn.citation_richness,
    fn.has_high_ref_count,
    
    -- Temporal features
    ft2.years_since_publication,
    ft2.publication_era,
    ft2.decade,
    ft2.is_recent,
    
    -- Quality features
    fq.citation_per_year,
    fq.is_highly_cited_for_year,
    fq.impact_score,
    
    -- Venue features
    fv.venue,
    fv.venue_median_citations,
    fv.venue_volume,
    fv.is_top_venue,
    fv.is_english,
    fv.venue_tier

FROM read_parquet('{PARQUET}') AS p
LEFT JOIN features_text ft ON p.id = ft.id
LEFT JOIN features_author fa ON p.id = fa.id
LEFT JOIN features_network fn ON p.id = fn.id
LEFT JOIN features_temporal ft2 ON p.id = ft2.id
LEFT JOIN features_quality fq ON p.id = fq.id
LEFT JOIN features_venue fv ON p.id = fv.id

WHERE p.n_citation IS NOT NULL
""".format(PARQUET=PARQUET)

con.sql(master_sql)

# Check how many features we created
row_count = con.sql("SELECT COUNT(*) FROM papers_with_features").df().iloc[0, 0]
col_count = len(con.sql("SELECT * FROM papers_with_features LIMIT 1").df().columns)

print(f"\n DONE MASTER FEATURES TABLE CREATED!")
print(f"   • Rows: {row_count:,} papers")
print(f"   • Columns: {col_count} features + metadata")

In [ ]:
TABLE_NAME = "papers_with_features_with_refs"

master_sql = f"""
CREATE OR REPLACE TABLE {TABLE_NAME} AS
SELECT 
    p.id,
    p.title,
    p.references,       
    p.year,
    p.n_citation AS target_variable,
    
    -- Text features
    ft.title_word_count,
    ft.abstract_word_count,
    ft.title_abstract_ratio,
    ft.title_complexity,
    ft.abstract_complexity,
    ft.has_numbers_in_title,
    ft.has_numbers_in_abstract,
    
    -- Author features
    fa.n_authors,
    fa.avg_author_citations,
    fa.max_author_citations,
    fa.multi_authored,
    
    -- Network features
    fn.n_references,
    fn.references_author_ratio,
    fn.citation_richness,
    fn.has_high_ref_count,
    
    -- Temporal features
    ft2.years_since_publication,
    ft2.publication_era,
    ft2.decade,
    ft2.is_recent,
    
    -- Quality features
    fq.citation_per_year,
    fq.is_highly_cited_for_year,
    fq.impact_score,
    
    -- Venue features
    fv.venue,
    fv.venue_median_citations,
    fv.venue_volume,
    fv.is_top_venue,
    fv.is_english,
    fv.venue_tier

FROM read_parquet('{PARQUET}') AS p
LEFT JOIN features_text ft ON p.id = ft.id
LEFT JOIN features_author fa ON p.id = fa.id
LEFT JOIN features_network fn ON p.id = fn.id
LEFT JOIN features_temporal ft2 ON p.id = ft2.id
LEFT JOIN features_quality fq ON p.id = fq.id
LEFT JOIN features_venue fv ON p.id = fv.id

WHERE p.n_citation IS NOT NULL
"""

con.sql(master_sql)

# Check
row_count = con.sql(f"SELECT COUNT(*) FROM {TABLE_NAME}").df().iloc[0, 0]
col_count = len(con.sql(f"SELECT * FROM {TABLE_NAME} LIMIT 1").df().columns)

print(f"\nDONE MASTER FEATURES TABLE CREATED!")
print(f"   • Rows: {row_count:,} papers")
print(f"   • Columns: {col_count} features + metadata")

# Salva parquet direttamente con DuckDB 
con.sql(f"COPY (SELECT * FROM {TABLE_NAME}) TO '{TABLE_NAME}.parquet' (FORMAT PARQUET)")
print(f"   • Saved: {TABLE_NAME}.parquet")

##  CORRELATION ANALYSIS WITH n_citations AS TARGET

In [ ]:
# Load features into pandas for correlation analysis
df_features = con.sql("""
SELECT *
FROM papers_with_features
USING SAMPLE 50000
""").df()

# Filter where target > 0
df_features = df_features[df_features['target_variable'] > 0]

print(f"Loaded {len(df_features):,} papers for analysis")

# Select numeric features
numeric_features = [
    'title_word_count', 'abstract_word_count', 'title_abstract_ratio',
    'title_complexity', 'abstract_complexity',
    'n_authors', 'avg_author_citations', 
    'max_author_citations',
    'n_references', 'references_author_ratio', 'citation_richness',
    'years_since_publication', 
    'citation_per_year', 'impact_score',
    'venue_median_citations', 'venue_volume', 'is_top_venue', 'is_english'
]

# Compute correlations
correlations = {}
for feat in numeric_features:
    if feat in df_features.columns:
        corr = df_features[[feat, 'target_variable']].corr().iloc[0, 1]
        correlations[feat] = corr

# Create dataframe and sort
corr_df = pd.DataFrame(list(correlations.items()), columns=['Feature', 'Correlation'])
corr_df = corr_df.sort_values('Correlation', key=abs, ascending=False)

print("\nTOP 15 MOST CORRELATED FEATURES WITH CITATIONS:")
print(corr_df.head(15).to_string(index=False))

# Visualize top correlations
fig, ax = plt.subplots(figsize=(10, 6))
top_corr = corr_df.head(15)
colors = ['#2ecc71' if x > 0 else '#e74c3c' for x in top_corr['Correlation']]
bars = ax.barh(range(len(top_corr)), top_corr['Correlation'], color=colors)
ax.set_yticks(range(len(top_corr)))
ax.set_yticklabels(top_corr['Feature'], fontsize=10)
ax.set_xlabel('Correlation with Citations', fontsize=11)
ax.set_title('Top 15 Features Predicting Citation Count', fontsize=13, fontweight='bold')
ax.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

# Feature importance summary
print("\n FEATURE CORRELATION SUMMARY:")
print(f"   Strong predictors (|corr| > 0.4): {len(corr_df[abs(corr_df['Correlation']) > 0.4])}")
print(f"   Moderate predictors (0.2 < |corr| ≤ 0.4): {len(corr_df[(abs(corr_df['Correlation']) > 0.2) & (abs(corr_df['Correlation']) <= 0.4)])}")
print(f"   Weak predictors (|corr| ≤ 0.2): {len(corr_df[abs(corr_df['Correlation']) <= 0.2])}")

In [ ]:
# df = con.sql(f"""
#     SELECT
#         ARRAY_LENGTH(authors) AS n_authors,
#         ARRAY_LENGTH(keywords) AS n_keywords,
#         ARRAY_LENGTH("references") AS n_references,
#         TRY_CAST(page_end AS INT) - TRY_CAST(page_start AS INT) AS page_count,
#         LENGTH(abstract) AS abstract_length,
#         n_citation
#     FROM read_parquet('{PARQUET}')
#     WHERE n_citation IS NOT NULL
#     USING SAMPLE 20000
# """).df().dropna()

# sns.heatmap(df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
# plt.title("Feature Correlations")
# plt.tight_layout()
# plt.show()

##  FEATURE INTERACTIONS & SEGMENTATION

In [ ]:
# 1. Venue Tier Impact
print(" VENUE TIER ANALYSIS")
venue_analysis = df_features.groupby('venue_tier').agg({
    'target_variable': ['count', 'mean', 'median', 'std'],
    'citation_per_year': 'mean',
    'n_references': 'mean'
}).round(2)
print(venue_analysis)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot
venue_order = ['elite', 'high', 'medium', 'low']
df_features_sorted = df_features.copy()
df_features_sorted['venue_tier'] = pd.Categorical(df_features_sorted['venue_tier'], categories=venue_order, ordered=True)

sns.boxplot(data=df_features_sorted, x='venue_tier', y='target_variable', ax=axes[0], palette='Set2')
axes[0].set_ylabel('Citations')
axes[0].set_title('Citations Distribution by Venue Tier')

sns.violinplot(data=df_features_sorted, x='venue_tier', y='citation_per_year', ax=axes[1], palette='Set3')
axes[1].set_ylabel('Citations Per Year (Normalized)')
axes[1].set_title('Citation Rate by Venue Tier')
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

# 2. Author Reputation Impact
print("\n👥 AUTHOR REPUTATION TIERS")
df_features['author_tier'] = pd.cut(df_features['max_author_citations'], 
                                     bins=[0, 5, 20, 100, 10000],
                                     labels=['Emerging', 'Mid-Career', 'Established', 'Star'])

author_analysis = df_features.groupby('author_tier').agg({
    'target_variable': ['count', 'mean', 'median'],
    'n_authors': 'mean'
}).round(2)
print(author_analysis)

df_features.boxplot(column='target_variable', by='author_tier', figsize=(10, 5))
plt.suptitle('')
plt.title('Citations Distribution by Author Career Stage')
plt.ylabel('Citations')
plt.show()

# 3. Era Analysis
print("\nTEMPORAL TRENDS")
era_analysis = df_features.groupby('publication_era').agg({
    'target_variable': ['count', 'mean', 'median'],
    'citation_per_year': 'mean'
}).round(2)
print(era_analysis)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

era_order = ['pre-2000', '2000-2009', '2010-2019', '2020+']
df_temp = df_features.copy()
df_temp['publication_era'] = pd.Categorical(df_temp['publication_era'], categories=era_order, ordered=True)
df_temp = df_temp.sort_values('publication_era')

sns.boxplot(data=df_temp, x='publication_era', y='target_variable', ax=ax1, palette='Set1')
ax1.set_ylabel('Citations')
ax1.set_title('Raw Citations by Era')

df_temp_norm = df_temp[df_temp['citation_per_year'].notna()]
sns.violinplot(data=df_temp_norm, x='publication_era', y='citation_per_year', ax=ax2, palette='Set2')
ax2.set_ylabel('Citations Per Year')
ax2.set_title('Normalized Citation Rate by Era')
ax2.set_yscale('log')

plt.tight_layout()
plt.show()

# 4. Complex Interactions
print("\n VENUE x AUTHOR INTERACTION")
interaction = df_features.groupby(['venue_tier', 'author_tier'])['target_variable'].agg(['count', 'mean']).round(1)
print(interaction)

In [ ]:
# COMPREHENSIVE CORRELATION MATRIX - All Features Combined
print("Building comprehensive correlation matrix with ALL features...")

# Load all features from papers_with_features
df_all_features = con.sql("""
SELECT 
    title_word_count,
    abstract_word_count,
    title_abstract_ratio,
    title_complexity,
    abstract_complexity,
    n_authors,
    avg_author_citations,
    max_author_citations,
    n_references,
    references_author_ratio,
    citation_richness,
    years_since_publication,
    citation_per_year,
    impact_score,
    venue_median_citations,
    venue_volume,
    is_top_venue,
    is_english,
    target_variable
FROM papers_with_features
USING SAMPLE 50000
""").df()

# Add features from the raw parquet (that might have additional info)
df_raw = con.sql(f"""
    SELECT
        ARRAY_LENGTH(authors) AS n_authors_raw,
        ARRAY_LENGTH(keywords) AS n_keywords,
        ARRAY_LENGTH("references") AS n_references_raw,
        TRY_CAST(page_end AS INT) - TRY_CAST(page_start AS INT) AS page_count,
        LENGTH(abstract) AS abstract_length,
        n_citation
    FROM read_parquet('{PARQUET}')
    WHERE n_citation IS NOT NULL
    USING SAMPLE 50000
""").df().dropna()

# Select numeric columns from raw that are not exact duplicates
# (n_authors and n_references are already in df_all_features, so we skip the _raw versions)
additional_features = df_raw[['n_keywords', 'page_count', 'abstract_length']].copy()

# Remove rows with NaN in the main dataframe
df_all_features = df_all_features.dropna()

# Merge if shapes allow (take intersection of indices)
if len(df_all_features) > 0 and len(additional_features) > 0:
    # Align sizes - take minimum
    min_size = min(len(df_all_features), len(additional_features))
    df_all_features = df_all_features.iloc[:min_size].reset_index(drop=True)
    additional_features = additional_features.iloc[:min_size].reset_index(drop=True)
    
    # Combine all features
    df_combined = pd.concat([df_all_features, additional_features], axis=1)
else:
    df_combined = df_all_features

print(f"Combined dataset shape: {df_combined.shape}")
print(f"Features included: {list(df_combined.columns)}")

# === FINAL COMPREHENSIVE CORRELATION MATRIX ===
# Calculate correlation matrix
correlation_matrix = df_combined.corr()

# Create large heatmap
fig, ax = plt.subplots(figsize=(18, 16))
sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={'label': 'Correlation Coefficient'},
    ax=ax,
    annot_kws={'size': 8}
)

ax.set_title('COMPREHENSIVE CORRELATION MATRIX - All Features', 
             fontsize=16, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.show()

# === CORRELATION WITH TARGET ===
print("\n" + "="*80)
print("CORRELATION WITH TARGET VARIABLE (n_citation)")
print("="*80)

target_corr = correlation_matrix['target_variable'].drop('target_variable').sort_values(key=abs, ascending=False)
print("\nAll features ranked by correlation strength with citations:")
for feat, corr_val in target_corr.items():
    stars = "★★★" if abs(corr_val) > 0.4 else "★★" if abs(corr_val) > 0.2 else "★"
    print(f"  {stars} {feat:30} {corr_val:+.4f}")

# === FEATURE GROUPING INSIGHTS ===
print("\n" + "="*80)
print("FEATURE STRENGTH SUMMARY")
print("="*80)
strong = len(target_corr[abs(target_corr) > 0.4])
moderate = len(target_corr[(abs(target_corr) > 0.2) & (abs(target_corr) <= 0.4)])
weak = len(target_corr[abs(target_corr) <= 0.2])

print(f"Strong predictors (|r| > 0.40):      {strong:2d} features")
print(f"Moderate predictors (0.2 < |r| ≤ 0.40): {moderate:2d} features")
print(f"Weak predictors (|r| ≤ 0.20):        {weak:2d} features")
print(f"Total features analyzed:             {len(target_corr):2d} features")

# === INTER-FEATURE CORRELATIONS ===
print("\n" + "="*80)
print("HIGH INTER-FEATURE CORRELATIONS (|r| > 0.7) - Potential Multicollinearity")
print("="*80)

# Find highly correlated feature pairs (excluding diagonal and duplicates)
corr_pairs = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        corr_val = correlation_matrix.iloc[i, j]
        if abs(corr_val) > 0.7:
            feat1 = correlation_matrix.columns[i]
            feat2 = correlation_matrix.columns[j]
            corr_pairs.append((feat1, feat2, corr_val))

corr_pairs = sorted(corr_pairs, key=lambda x: abs(x[2]), reverse=True)

if corr_pairs:
    for f1, f2, corr in corr_pairs[:15]:  # Top 15
        print(f"  {f1:25} <-> {f2:25} | r = {corr:+.4f}")
else:
    print("  No highly correlated feature pairs found (r > 0.7)")

print("\n" + "="*80)

## EXPORT & SUMMARY

Esportiamo i risultati del feature engineering per uso in modeling:

In [ ]:
# Export complete feature set
con.sql("COPY papers_with_features TO 'papers_with_features.parquet' (FORMAT PARQUET)")
print(" Exported: papers_with_features.parquet")

# Export top cited papers
con.sql("""
COPY (
    SELECT * FROM papers_with_features
    ORDER BY target_variable DESC
    LIMIT 100
) TO 'top_cited_papers_features.parquet' (FORMAT PARQUET)
""")
print("Exported: top_cited_papers_features.parquet")

# Generate summary report
print("\n" + "="*80)
print("FEATURE ENGINEERING FINAL REPORT")
print("="*80)

print(f"\nTotal Papers with Features: {len(df_features):,}")
print(f"Total Features Created: {len(numeric_features)} numeric + 5 categorical")
print(f"\n TOP 5 PREDICTIVE FEATURES:")
for i, row in corr_df.head(5).iterrows():
    print(f"   {i+1}. {row['Feature']:30} | Corr: {row['Correlation']:+.4f}")

print(f"\n KEY INSIGHTS:")
print(f"   • Strong venues get {df_features[df_features['venue_tier']=='elite']['target_variable'].mean():.0f} citations vs {df_features[df_features['venue_tier']=='low']['target_variable'].mean():.0f} for weak")
print(f"   • Papers with star authors: {df_features[df_features['max_author_citations']>100]['target_variable'].mean():.0f} cit. vs {df_features[df_features['max_author_citations']<=5]['target_variable'].mean():.0f} for emerging")
print(f"   • Pre-2000 papers: {df_features[df_features['year']<2000]['target_variable'].mean():.0f} citations (mature)")
print(f"   • 2020+ papers: {df_features[df_features['year']>=2020]['target_variable'].mean():.0f} citations (incomplete)")

print("\n" + "="*80)

In [ ]:
# %pip install -q torch torchvision torchaudio
# %pip install -q torch-geometric scikit-learn

# TRAINING

In [ ]:
# import os
# import torch
# import numpy as np
# import pandas as pd
# from sklearn.preprocessing import StandardScaler
# from torch_geometric.utils import to_undirected


# REBUILD_SPLITS = False


# SPLIT_DIR   = "splits_gcn"
# NODE_PATH   = f"{SPLIT_DIR}/node_df.parquet"
# EDGE_PATH   = f"{SPLIT_DIR}/edge_df.parquet"
# SCALER_MEAN = f"{SPLIT_DIR}/scaler_mean.npy"
# SCALER_STD  = f"{SPLIT_DIR}/scaler_std.npy"


# num_cols = [
#     'n_authors', 'avg_author_citations', 'max_author_citations',
#     'n_references', 'references_author_ratio', 'citation_richness',
#     'years_since_publication', 'citation_per_year', 'impact_score',
#     'venue_median_citations', 'venue_volume', 'is_top_venue', 'is_english'
# ]


# if REBUILD_SPLITS or not os.path.exists(NODE_PATH):
#     os.makedirs(SPLIT_DIR, exist_ok=True)

#     sample_n = 15000

#     # 1. Sample nodes with a fixed seed for reproducibility
#     node_df = con.sql(f"""
#         SELECT id, year,
#                n_authors, avg_author_citations, max_author_citations,
#                n_references, references_author_ratio, citation_richness,
#                years_since_publication, citation_per_year, impact_score,
#                venue_median_citations, venue_volume, is_top_venue, is_english
#         FROM papers_with_features
#         WHERE id IS NOT NULL AND TRIM(id) <> ''
#           AND year IS NOT NULL
#         USING SAMPLE reservoir({sample_n})
#     """).df().reset_index(drop=True)

#     node_df['node_idx'] = node_df.index
#     con.register("sampled_papers", node_df[['id']])

#     # 2. Retrieve edges between sampled papers
#     edge_df = con.sql("""
#         SELECT p.id AS src_id, ref AS dst_id
#         FROM papers p
#         CROSS JOIN UNNEST(p."references") AS t(ref)
#         JOIN sampled_papers sp_src ON p.id  = sp_src.id
#         JOIN sampled_papers sp_dst ON ref   = sp_dst.id
#     """).df().drop_duplicates().reset_index(drop=True)

#     if edge_df.empty:
#         raise RuntimeError("No citations found among the sampled papers.")

#     edge_df = (edge_df
#         .merge(node_df[['id', 'node_idx']], left_on='src_id', right_on='id')
#         .rename(columns={'node_idx': 'src_idx'}).drop(columns=['id'])
#         .merge(node_df[['id', 'node_idx']], left_on='dst_id', right_on='id')
#         .rename(columns={'node_idx': 'dst_idx'}).drop(columns=['id'])
#     )

#     # 3. Temporal split on edges — prevents data leakage
#     edge_df = edge_df.merge(
#         node_df[['node_idx', 'year']].rename(columns={'node_idx': 'src_idx', 'year': 'src_year'}),
#         on='src_idx'
#     )

#     edge_df = edge_df.sort_values('src_year').reset_index(drop=True)
#     n         = len(edge_df)
#     train_end = int(n * 0.70)
#     val_end   = int(n * 0.85)

#     conditions = [
#         edge_df.index < train_end,
#         (edge_df.index >= train_end) & (edge_df.index < val_end),
#     ]
#     edge_df['split'] = np.select(conditions, ['train', 'val'], default='test')

#     print(f"Total edges: {n}")
#     print(edge_df['split'].value_counts().sort_index())
#     print(f"Max year train: {edge_df[edge_df.split=='train']['src_year'].max()}")
#     print(f"Min year val:   {edge_df[edge_df.split=='val']['src_year'].min()}")
#     print(f"Min year test:  {edge_df[edge_df.split=='test']['src_year'].min()}")

#     # 4. Normalisation — fit only on train nodes
#     train_node_ids = set(
#         edge_df[edge_df.split == 'train']['src_idx'].tolist() +
#         edge_df[edge_df.split == 'train']['dst_idx'].tolist()
#     )
#     train_mask = node_df['node_idx'].isin(train_node_ids)

#     features_raw = node_df[num_cols].to_numpy(dtype=float)
#     features_raw = np.nan_to_num(features_raw, nan=0.0, posinf=1e9, neginf=-1e9)

#     scaler = StandardScaler()
#     scaler.fit(features_raw[train_mask])
#     node_df[num_cols] = scaler.transform(features_raw)

#     # 5. Save
#     node_df.to_parquet(NODE_PATH, index=False)
#     edge_df.to_parquet(EDGE_PATH, index=False)
#     np.save(SCALER_MEAN, scaler.mean_)
#     np.save(SCALER_STD,  scaler.scale_)
#     print("Splits saved to", SPLIT_DIR)

# else:
#     print("Existing splits found, loading...")
#     node_df = pd.read_parquet(NODE_PATH)
#     edge_df = pd.read_parquet(EDGE_PATH)
#     print(f"Nodes: {len(node_df)} | Edges: {len(edge_df)}")
#     print(edge_df['split'].value_counts().sort_index())

## SPLIT

In [1]:
import pyarrow.parquet as pq
import pandas as pd

INPUT_FILE = "papers_with_features_with_refs.parquet"

# Read schema without loading data
pf = pq.read_schema(INPUT_FILE)
print(f"Total columns: {len(pf.names)}\n")
print(f"{'Column':<45} {'Type':<20}")
print("-" * 65)
for name in pf.names:
    print(f"{name:<45} {str(pf.field(name).type):<20}")

# Load only first 3 rows to see actual values
print("\n--- Sample values (3 rows) ---")
sample = pd.read_parquet(INPUT_FILE).head(3)
for col in sample.columns:
    print(f"\n{col}: {sample[col].tolist()}")

Total columns: 33

Column                                        Type                
-----------------------------------------------------------------
id                                            string              
title                                         string              
references                                    list<element: string>
year                                          int64               
target_variable                               int64               
title_word_count                              int64               
abstract_word_count                           int64               
title_abstract_ratio                          float               
title_complexity                              float               
abstract_complexity                           float               
has_numbers_in_title                          int32               
has_numbers_in_abstract                       int32               
n_authors                                  

In [2]:
leaky_features = [
    "citation_per_year",
    "impact_score",
    "citation_richness",
    "is_highly_cited_for_year",
    "avg_author_citations",
    "max_author_citations",
    "venue_median_citations",
    "years_since_publication",
    "n_references",           
    "references_author_ratio",
    "has_high_ref_count",     
]

In [3]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║                        CONFIGURATION                                    ║
# ╚══════════════════════════════════════════════════════════════════════════╝
INPUT_FILE = "papers_with_features_with_refs.parquet"  # ← file con references
OUTPUT_DIR = "model_splits"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Check if split files already exist
split_files_exist = all([
    os.path.exists(f"{OUTPUT_DIR}/train.parquet"),
    os.path.exists(f"{OUTPUT_DIR}/val.parquet"),
    os.path.exists(f"{OUTPUT_DIR}/test.parquet")
])

if split_files_exist:
    print(f"Split files already exist in '{OUTPUT_DIR}'. Skipping creation.")
else:
    print("Loading dataset...")
    df = pd.read_parquet(INPUT_FILE)

    # ── 1. Temporal sorting ───────────────────────────────────────────────
    df = df.dropna(subset=["year"]).copy()
    df = df.sort_values("year").reset_index(drop=True)

    # Sequential node_idx 0..N-1 (required by PyTorch Geometric)
    df["node_idx"] = np.arange(len(df), dtype=np.int32)

    # ── 2. Feature definition (NO TARGET NODE FEATURE PER LA LINK PREDICTION)
    
    # Lista di feature da ignorare (leaky_features va definita prima, es. leaky_features = [])
    leaky_features = getattr(globals(), 'leaky_features', []) 

    # Colonne strutturali che NON devono essere usate come feature numeriche (non vanno scalate)
    non_feature_cols = [
        "id",
        "n_citation",
        "node_idx",
        "split",
        "year",
        "title",
        "abstract",
        "venue",
        "publication_era",
        "venue_tier",
        "references",       # ← FONDAMENTALE: contiene i link (edge_index)
        "target_variable"   # Lo includiamo qui solo per evitare che venga scalato se fosse rimasto nel dataframe per sbaglio
    ]

    exclude_cols = set(non_feature_cols + list(leaky_features))

    # Seleziona solo le colonne numeriche valide
    safe_numeric_features = [
        col for col in df.columns
        if col not in exclude_cols
        and pd.api.types.is_numeric_dtype(df[col])
    ]

    print(f"\nSafe numeric features selected ({len(safe_numeric_features)}):")
    for f in safe_numeric_features[:5]:
        print(f"  ✓  {f}")
    if len(safe_numeric_features) > 5:
        print("  ... e altre")

    # ── 3. Temporal splitting (Train <= 2020, Val 2021-2022, Test >= 2023) ───────────────
    df["split"] = ""
    
    df.loc[df["year"] <= 2020, "split"] = "train"
    df.loc[(df["year"] >= 2021) & (df["year"] <= 2022), "split"] = "val"
    df.loc[df["year"] >= 2023, "split"] = "test"

    print("\nSplit distribution:")
    print(df["split"].value_counts().reindex(["train", "val", "test"]))
    print(f"  Train years : {df.loc[df['split']=='train', 'year'].min()} – "
          f"{df.loc[df['split']=='train', 'year'].max()}")
    print(f"  Val years   : {df.loc[df['split']=='val',   'year'].min()} – "
          f"{df.loc[df['split']=='val',   'year'].max()}")
    print(f"  Test years  : {df.loc[df['split']=='test',  'year'].min()} – "
          f"{df.loc[df['split']=='test',  'year'].max()}")

    # ── 4. Strict scaling (fit ONLY on train, transform val/test) ────────
    X_train = df.loc[df["split"] == "train", safe_numeric_features].to_numpy()
    X_val   = df.loc[df["split"] == "val",   safe_numeric_features].to_numpy()
    X_test  = df.loc[df["split"] == "test",  safe_numeric_features].to_numpy()

    X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)
    X_val   = np.nan_to_num(X_val,   nan=0.0, posinf=0.0, neginf=0.0)
    X_test  = np.nan_to_num(X_test,  nan=0.0, posinf=0.0, neginf=0.0)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled   = scaler.transform(X_val)
    X_test_scaled  = scaler.transform(X_test)

    # Cast a float64 per sicurezza e assegnazione
    for col in safe_numeric_features:
        df[col] = df[col].astype("float64")

    df.loc[df["split"] == "train", safe_numeric_features] = X_train_scaled
    df.loc[df["split"] == "val",   safe_numeric_features] = X_val_scaled
    df.loc[df["split"] == "test",  safe_numeric_features] = X_test_scaled

    # ── 5. Save files ─────────────────────────────────────────────────────
    
    # Colonne da mantenere nel file salvato (features + struttura + link)
    keep_cols = safe_numeric_features + [
        "id",
        "node_idx",
        "split",
        "year",
        "references",       # ← vitale per costruire gli edge_index (archi) successivamente
    ]

    # Sicurezza extra: rimuove eventuali colonne duplicate mantenendo l'ordine
    keep_cols = list(dict.fromkeys(keep_cols))

    print("\nSaving split files...")
    df.loc[df["split"] == "train", keep_cols].to_parquet(
        f"{OUTPUT_DIR}/train.parquet", index=False)
    df.loc[df["split"] == "val",   keep_cols].to_parquet(
        f"{OUTPUT_DIR}/val.parquet",   index=False)
    df.loc[df["split"] == "test",  keep_cols].to_parquet(
        f"{OUTPUT_DIR}/test.parquet",  index=False)

    np.save(f"{OUTPUT_DIR}/scaler_mean.npy",  scaler.mean_)
    np.save(f"{OUTPUT_DIR}/scaler_scale.npy", scaler.scale_)

    print(f"\n✓ Done. Files saved in '{OUTPUT_DIR}/'")
    print(f"   Node Features        : {len(safe_numeric_features)}")
    print(f"   Structural columns   : id, node_idx, split, year")
    print(f"   Graph Edges column   : references")
    print(f"   Scaler saved         : scaler_mean.npy, scaler_scale.npy")

Split files already exist in 'model_splits'. Skipping creation.


In [4]:
# import pyarrow.parquet as pq

# # Reread LOAD_COLS from the current schema of the new Parquet files
# pf_new = pq.read_schema(f"{OUTPUT_DIR}/train.parquet")
# LOAD_COLS = [c for c in pf_new.names if c not in {"split"}]

# print("LOAD_COLS updated:", LOAD_COLS)

# train_df = load_and_sample(f"{OUTPUT_DIR}/train.parquet", TRAIN_SAMPLE, LOAD_COLS)
# val_df   = load_and_sample(f"{OUTPUT_DIR}/val.parquet",   VAL_SAMPLE,   LOAD_COLS)
# test_df  = load_and_sample(f"{OUTPUT_DIR}/test.parquet",  TEST_SAMPLE,  LOAD_COLS)

# train_df

# CELLA DA ESEGUIRE PER CARICARE GLI SPLIT

In [5]:
import polars as pl

# Carica i file parquet come DataFrame Polars
train_df = pl.read_parquet('model_splits/train.parquet')
val_df = pl.read_parquet('model_splits/val.parquet')
test_df = pl.read_parquet('model_splits/test.parquet')

train_df

title_word_count,abstract_word_count,title_abstract_ratio,title_complexity,abstract_complexity,has_numbers_in_title,has_numbers_in_abstract,n_authors,avg_author_citations,max_author_citations,multi_authored,n_references,references_author_ratio,citation_richness,has_high_ref_count,years_since_publication,decade,is_recent,citation_per_year,is_highly_cited_for_year,impact_score,venue_median_citations,venue_volume,is_top_venue,is_english,id,node_idx,split,year,references
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,i32,str,i64,list[str]
1.889193,0.966761,-0.097798,-1.158418,-0.017696,0.0,0.0,0.343616,-0.205446,-0.188856,0.422509,-0.805261,-0.650567,-0.602605,-0.092262,181.828368,-181.828368,-0.4163,-0.095382,-1.021094,-0.094982,-0.360159,-0.228457,-0.003601,-5.189611,"""60376557d3485cfff1d91b49""",0,"""train""",0,[]
-1.307626,-0.161032,-0.35108,0.991659,0.282907,0.0,0.0,-0.457601,-0.211314,-0.196218,0.422509,-0.805261,-0.650567,-0.602605,-0.092262,181.828368,-181.828368,-0.4163,-0.095295,0.979342,-0.094958,-0.360159,-0.228457,-0.003601,-5.189611,"""5e54f1813a55acae32a25d83""",1,"""train""",0,[]
-0.508422,0.041717,-0.206753,0.280939,0.152899,0.0,0.0,0.744225,-0.272097,-0.197655,0.422509,-0.805261,-0.650567,-0.602605,-0.092262,181.828368,-181.828368,-0.4163,-0.095394,-1.021094,-0.094985,-0.360159,-0.228457,-0.003601,-5.189611,"""5e4d083f3a55ac8cfd770a60""",2,"""train""",0,[]
-1.041225,-0.541187,-0.356683,-1.715845,0.300257,0.0,0.0,0.744225,-0.257661,-0.195859,0.422509,-0.805261,-0.650567,-0.602605,-0.092262,181.828368,-181.828368,-0.4163,-0.095382,-1.021094,-0.094982,0.483989,-0.232516,-0.003601,-5.189611,"""5e79da3b91e0115bb11579c1""",3,"""train""",0,[]
-1.041225,0.561262,-0.436269,0.450158,0.232997,0.0,0.0,1.545442,0.400041,6.527496,0.422509,-0.805261,-0.650567,-0.602605,-0.092262,181.828368,-181.828368,-0.4163,-0.093368,0.979342,-0.094432,0.192374,-0.23262,-0.003601,-5.189611,"""5dce788a3a55ac9580a16242""",4,"""train""",0,[]
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
0.024382,0.599278,-0.26556,-0.307943,0.104951,0.0,0.0,-0.457601,-0.198442,-0.186701,0.422509,-0.633791,-0.46822,-0.508719,-0.092262,-0.89694,0.89694,2.402113,-0.070286,-1.021094,-0.071551,-0.26807,-0.231839,-0.003601,0.192693,"""5ea0135c9fced0a24b9d88e4""",4087964,"""train""",2020,"[""53909f8c20f70186a0e3fe15"", ""5550484a45ce0a409eb6d288""]"
0.290783,0.13042,-0.070835,0.105567,0.185384,0.0,0.0,0.343616,-0.122447,-0.042509,0.422509,-0.46232,-0.46822,-0.377705,-0.092262,-0.89694,0.89694,2.402113,-0.070286,-1.021094,-0.071551,-0.053196,-0.187034,-0.003601,0.192693,"""5ef876f191e0115941835f89""",4087965,"""train""",2020,"[""5a9cb60d17c44a376ffb3a5b"", ""5a260c3b17c44a4ba8a25fdb"", … ""57d063f1ac4436735429616e""]"
0.823587,-0.819967,0.735907,-0.195478,0.257849,0.0,0.0,0.343616,-0.158445,-0.183469,0.422509,-0.205114,-0.33146,0.166356,-0.092262,-0.89694,0.89694,2.402113,-0.01368,-1.021094,-0.018795,-0.191329,-0.200564,-0.003601,0.192693,"""5ea014de9fced0a24ba1804a""",4087966,"""train""",2020,"[""53e998f0b7602d970212b267"", ""5550404845ce0a409eb334a6"", … ""5c2ff92adf5b8c0b3cee7e91""]"


## Preparation for LGBM and XGBoost

In [ ]:
pip install torch

In [6]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║               CELLA 0 — COSTRUZIONE DATI CONDIVISI                     ║
# ║  Questa cella va eseguita UNA SOLA VOLTA prima di LGBM e XGBoost.      ║
# ║  Produce: X_train_xgb, X_val_xgb, X_test_xgb,                         ║
# ║           y_train_xgb, y_val_xgb, y_test_xgb,                         ║
# ║           feature_names_xgb                                            ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import os
import polars as pl
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURAZIONE: quale percentuale di archi usare per ogni split
# Riduci se hai poca RAM, aumenta per metriche più stabili
# ─────────────────────────────────────────────────────────────────────────────
FRACTION_TRAIN = 1   # 5% degli archi di training   → ~1.5M coppie
FRACTION_VAL   = 1   # 10% degli archi di val       → ~0.9M coppie
FRACTION_TEST  = 1   # 10% degli archi di test      → ~1.0M coppie


# ─────────────────────────────────────────────────────────────────────────────
# SEZIONE 0: Caricamento dei file parquet già pre-processati e scalati
# (lo scaling StandardScaler è stato fatto fit SOLO su train nel preprocessing)
# ─────────────────────────────────────────────────────────────────────────────
print("Loading Parquet splits...")

# Concatena tutto per costruire la feature matrix globale e il mapping id→idx
# NON si usa all_df per scalare (quello è già fatto) — serve solo per indicizzare
all_df = pl.concat([train_df, val_df, test_df]).sort("node_idx")

# Soglie temporali: num_train_nodes è il numero di paper esistenti fino al 2020
# Serve per evitare che il training veda paper "del futuro"
num_train_nodes = int(train_df["node_idx"].max()) + 1
num_val_nodes   = int(max(train_df["node_idx"].max(), val_df["node_idx"].max())) + 1
num_total_nodes = int(all_df["node_idx"].max()) + 1

print(f"  Nodi train : {num_train_nodes:,}")
print(f"  Nodi val   : {num_val_nodes:,}")
print(f"  Nodi totali: {num_total_nodes:,}")

# Mapping da ID stringa (es. "abc123") a node_idx intero (es. 4521)
# Serve per risolvere le citazioni: "paper A cita paper B (id stringa)" → arco numerico
id_to_idx = all_df.select(["id", "node_idx"])


# ─────────────────────────────────────────────────────────────────────────────
# SEZIONE 1: Feature matrix dei nodi
# Ogni riga = un paper, ogni colonna = una feature numerica (25 in totale)
# Esempio di feature: impact_score, n_references, years_since_publication, ...
# ─────────────────────────────────────────────────────────────────────────────
exclude_cols = ["id", "node_idx", "split", "year", "references", "target_variable"]
feature_cols = [c for c in all_df.columns if c not in exclude_cols]

print(f"\nBuilding feature matrix: {num_total_nodes:,} nodi × {len(feature_cols)} feature...")
node_feats_full = all_df.select(feature_cols).to_numpy()
node_feats_full = np.nan_to_num(
    node_feats_full, nan=0.0, posinf=0.0, neginf=0.0
).astype(np.float32)

# FIX leakage: ogni split vede solo le feature dei nodi "già esistenti"
# node_feats_train[i] = feature del paper i, ma solo per i < num_train_nodes
# Questo impedisce al training di accedere a feature di paper futuri
node_feats_train = node_feats_full[:num_train_nodes]   # solo paper ≤ 2020
node_feats_val   = node_feats_full[:num_val_nodes]     # paper ≤ 2022
node_feats_test  = node_feats_full[:num_total_nodes]   # tutti i paper


# ─────────────────────────────────────────────────────────────────────────────
# SEZIONE 2: Estrazione archi positivi (citazioni reali)
# Un arco (src → dst) significa "il paper src cita il paper dst"
# FIX leakage: filtriamo archi il cui dst è un paper "futuro" rispetto allo split
# Es: un paper del 2022 NON può citare un paper del 2023 nell'esperimento di val
# ─────────────────────────────────────────────────────────────────────────────
def get_edge_index(split_df, max_dst_node):
    edges = (
        split_df.select(["node_idx", "references"])
        .explode("references")          # ogni citazione diventa una riga separata
        .drop_nulls()
        .rename({"node_idx": "src", "references": "target_id"})
    )
    # Risolvi l'ID stringa del paper citato nel suo node_idx numerico
    edges = edges.join(id_to_idx, left_on="target_id", right_on="id", how="inner")
    edges = edges.rename({"node_idx": "dst"})
    # Scarta archi verso paper "del futuro" rispetto a questo split
    edges = edges.filter(pl.col("dst") < max_dst_node)
    src_np = edges["src"].to_numpy()
    dst_np = edges["dst"].to_numpy()
    return torch.tensor(np.vstack([src_np, dst_np]), dtype=torch.long)


def subsample_edges(edge_index, fraction):
    """Prende un sottoinsieme casuale degli archi per risparmiare RAM."""
    if fraction >= 1.0:
        return edge_index
    n = max(1, int(edge_index.size(1) * fraction))
    return edge_index[:, torch.randperm(edge_index.size(1))[:n]]


print("Extracting positive edges (citazioni reali, filtrate temporalmente)...")
train_pos_full = get_edge_index(train_df, max_dst_node=num_train_nodes)
val_pos_full   = get_edge_index(val_df,   max_dst_node=num_val_nodes)
test_pos_full  = get_edge_index(test_df,  max_dst_node=num_total_nodes)

print("Applying subsampling...")
torch.manual_seed(42)
train_pos = subsample_edges(train_pos_full, FRACTION_TRAIN)
val_pos   = subsample_edges(val_pos_full,   FRACTION_VAL)
test_pos  = subsample_edges(test_pos_full,  FRACTION_TEST)

print(f"  Train: {train_pos.size(1):,} archi (su {train_pos_full.size(1):,} totali)")
print(f"  Val  : {val_pos.size(1):,} archi (su {val_pos_full.size(1):,} totali)")
print(f"  Test : {test_pos.size(1):,} archi (su {test_pos_full.size(1):,} totali)")


# ─────────────────────────────────────────────────────────────────────────────
# SEZIONE 3: Negative sampling temporale
# Per ogni arco positivo (citazione reale) generiamo un arco negativo (coppia falsa)
# → dataset bilanciato 50/50 per la classificazione binaria
#
# FIX leakage: i negativi di train NON possono essere archi reali di val/test
# Usiamo un set Python di codici int64 per il lookup O(1) (evita OOM di np.isin)
# Codifica: src * num_nodes + dst → int64 unico per ogni coppia
# ─────────────────────────────────────────────────────────────────────────────
def sample_negatives_fast(known_edge_index, num_nodes, target_count, oversample=2.0, seed=42):
    rng = np.random.default_rng(seed)
    src_k = known_edge_index[0].numpy().astype(np.int64)
    dst_k = known_edge_index[1].numpy().astype(np.int64)
    known_codes = set((src_k * num_nodes + dst_k).tolist())  # archi da escludere

    collected_src, collected_dst = [], []
    remaining = target_count

    while remaining > 0:
        # Sovracampiona (×2) per compensare quelli che verranno scartati
        n_sample = int(remaining * oversample)
        src_rand = rng.integers(0, num_nodes, size=n_sample, dtype=np.int64)
        dst_rand = rng.integers(0, num_nodes, size=n_sample, dtype=np.int64)
        valid = src_rand != dst_rand                            # scarta self-loops
        src_rand, dst_rand = src_rand[valid], dst_rand[valid]
        codes = (src_rand * num_nodes + dst_rand).tolist()
        mask = np.array([c not in known_codes for c in codes], dtype=bool)  # scarta archi reali
        src_rand, dst_rand = src_rand[mask], dst_rand[mask]
        take = min(remaining, len(src_rand))
        collected_src.append(src_rand[:take])
        collected_dst.append(dst_rand[:take])
        remaining -= take

    src_out = np.concatenate(collected_src).astype(np.int64)
    dst_out = np.concatenate(collected_dst).astype(np.int64)
    return torch.tensor(np.vstack([src_out, dst_out]), dtype=torch.long)


print("\nGenerating negative samples (temporalmente corretti)...")

neg_train = sample_negatives_fast(
    train_pos_full, num_nodes=num_train_nodes,
    target_count=train_pos.size(1), seed=42,
)
assert neg_train.size(1) == train_pos.size(1), "neg_train insufficienti — aumenta oversample"
print(f"  neg_train: {neg_train.size(1):,}")

neg_val = sample_negatives_fast(
    torch.cat([train_pos_full, val_pos_full], dim=1),
    num_nodes=num_val_nodes,
    target_count=val_pos.size(1), seed=43,
)
assert neg_val.size(1) == val_pos.size(1), "neg_val insufficienti — aumenta oversample"
print(f"  neg_val  : {neg_val.size(1):,}")

neg_test = sample_negatives_fast(
    torch.cat([train_pos_full, val_pos_full, test_pos_full], dim=1),
    num_nodes=num_total_nodes,
    target_count=test_pos.size(1), seed=44,
)
assert neg_test.size(1) == test_pos.size(1), "neg_test insufficienti — aumenta oversample"
print(f"  neg_test : {neg_test.size(1):,}")

# ─────────────────────────────────────────────────────────────────────────
# SEZIONE 4: Scrittura su disco delle matrici tabellari (MEMORY-SAFE)
#
# Invece di materializzare X_train_xgb (potenzialmente 15+ GB) in RAM,
# scriviamo le coppie feature in file numpy memmap a blocchi.
# Il sistema operativo gestirà il paging automaticamente.
#
# Per ogni arco (src → dst) creiamo una riga da N_FEAT*2 valori:
#   [feat_1_src, ..., feat_N_src | feat_1_dst, ..., feat_N_dst]
# Labels: 1 = citazione reale, 0 = coppia falsa
# ─────────────────────────────────────────────────────────────────────────
import gc

PAIRS_DIR  = "model_splits/pairs"
CHUNK_SIZE = 500_000   # righe per chunk — ~100 MB per chunk

os.makedirs(PAIRS_DIR, exist_ok=True)

def write_pairs_memmap(path_X, path_y, pos_edges, neg_edges, node_feats,
                       chunk_size=CHUNK_SIZE):
    """Scrive le feature delle coppie su disco in blocchi per evitare OOM."""
    n_pos   = pos_edges.size(1)
    n_neg   = neg_edges.size(1)
    n_total = n_pos + n_neg
    n_cols  = node_feats.shape[1] * 2        # src feats + dst feats

    # Crea file memmap (allocazione lazy, non occupa RAM)
    X_mm = np.memmap(path_X, dtype="float32", mode="w+", shape=(n_total, n_cols))
    y_mm = np.memmap(path_y, dtype="int32",   mode="w+", shape=(n_total,))

    offset = 0

    # ── Scrivi coppie positive ──
    for start in range(0, n_pos, chunk_size):
        end = min(start + chunk_size, n_pos)
        src = pos_edges[0, start:end].numpy()
        dst = pos_edges[1, start:end].numpy()
        X_mm[offset:offset + (end - start)] = np.hstack(
            [node_feats[src], node_feats[dst]]
        )
        y_mm[offset:offset + (end - start)] = 1
        offset += end - start

    # ── Scrivi coppie negative ──
    for start in range(0, n_neg, chunk_size):
        end = min(start + chunk_size, n_neg)
        src = neg_edges[0, start:end].numpy()
        dst = neg_edges[1, start:end].numpy()
        X_mm[offset:offset + (end - start)] = np.hstack(
            [node_feats[src], node_feats[dst]]
        )
        y_mm[offset:offset + (end - start)] = 0
        offset += end - start

    X_mm.flush()
    y_mm.flush()
    del X_mm, y_mm
    gc.collect()
    return n_total, n_cols


# ── Controlla se i file esistono già ──
pair_files_exist = all(
    os.path.exists(f"{PAIRS_DIR}/{f}")
    for f in ["X_train.dat", "y_train.dat",
              "X_val.dat",   "y_val.dat",
              "X_test.dat",  "y_test.dat",
              "shapes.json"]
)

if pair_files_exist:
    import json as _json
    pair_shapes = _json.load(open(f"{PAIRS_DIR}/shapes.json"))
    print(f"Pair files already exist in '{PAIRS_DIR}/'. Skipping creation.")
    print(f"  Train : {pair_shapes['train'][0]:,} × {pair_shapes['train'][1]}")
    print(f"  Val   : {pair_shapes['val'][0]:,}   × {pair_shapes['val'][1]}")
    print(f"  Test  : {pair_shapes['test'][0]:,}  × {pair_shapes['test'][1]}")
else:
    print("\nWriting pair features to disk (chunked, memory-safe)...")

    n_train, n_feat = write_pairs_memmap(
        f"{PAIRS_DIR}/X_train.dat", f"{PAIRS_DIR}/y_train.dat",
        train_pos, neg_train, node_feats_train,
    )
    print(f"  X_train : {n_train:,} × {n_feat}")

    # Libera tensori non più necessari
    del train_pos, neg_train, train_pos_full
    gc.collect()

    n_val, _ = write_pairs_memmap(
        f"{PAIRS_DIR}/X_val.dat", f"{PAIRS_DIR}/y_val.dat",
        val_pos, neg_val, node_feats_val,
    )
    print(f"  X_val   : {n_val:,} × {n_feat}")

    del val_pos, neg_val, val_pos_full
    gc.collect()

    n_test, _ = write_pairs_memmap(
        f"{PAIRS_DIR}/X_test.dat", f"{PAIRS_DIR}/y_test.dat",
        test_pos, neg_test, node_feats_test,
    )
    print(f"  X_test  : {n_test:,} × {n_feat}")

    del test_pos, neg_test, test_pos_full
    gc.collect()

    # Salva le dimensioni per caricamento successivo
    import json as _json
    pair_shapes = {
        "train": [n_train, n_feat],
        "val":   [n_val,   n_feat],
        "test":  [n_test,  n_feat],
    }
    with open(f"{PAIRS_DIR}/shapes.json", "w") as f:
        _json.dump(pair_shapes, f)

    print(f"\n✓ Pair data saved to '{PAIRS_DIR}/'")

# Nomi delle feature (src_* + dst_*)
feature_names_xgb = [f"src_{c}" for c in feature_cols] + [f"dst_{c}" for c in feature_cols]

print(f"\nFeature names ({len(feature_names_xgb)}): {feature_names_xgb[:4]} ... {feature_names_xgb[-2:]}")
print("Dati pronti. Esegui ora le celle LGBM e XGBoost.")


# ─────────────────────────────────────────────────────────────────────────────
# SEZIONE 5: Pulizia aggressiva della memoria
# Libera TUTTO ciò che non serve più (DataFrames, features, tensori)
# per massimizzare la RAM disponibile per LGBM/XGBoost
# ─────────────────────────────────────────────────────────────────────────────
import gc

_vars_to_free = [
    'train_df', 'val_df', 'test_df', 'all_df',
    'node_feats_full', 'node_feats_train', 'node_feats_val', 'node_feats_test',
    'id_to_idx',
    'train_pos', 'val_pos', 'test_pos',
    'train_pos_full', 'val_pos_full', 'test_pos_full',
    'neg_train', 'neg_val', 'neg_test',
]

_freed = []
for _v in _vars_to_free:
    if _v in dir():
        try:
            exec(f"del {_v}")
            _freed.append(_v)
        except:
            pass

gc.collect()
print(f"\n🧹 Memory cleanup: freed {len(_freed)} objects: {', '.join(_freed)}")
print(f"   Kept: pair_shapes, feature_cols, feature_names_xgb")


Loading Parquet splits...
  Nodi train : 4,087,969
  Nodi val   : 4,725,828
  Nodi totali: 5,590,897

Building feature matrix: 5,590,897 nodi × 25 feature...
Extracting positive edges (citazioni reali, filtrate temporalmente)...
Applying subsampling...
  Train: 31,760,720 archi (su 31,760,720 totali)
  Val  : 8,876,075 archi (su 8,876,075 totali)
  Test : 9,961,737 archi (su 9,961,737 totali)

Generating negative samples (temporalmente corretti)...
  neg_train: 31,760,720
  neg_val  : 8,876,075
  neg_test : 9,961,737
Pair files already exist in 'model_splits/pairs/'. Skipping creation.
  Train : 63,521,440 × 50
  Val   : 17,752,150   × 50
  Test  : 19,923,474  × 50

Feature names (50): ['src_title_word_count', 'src_abstract_word_count', 'src_title_abstract_ratio', 'src_title_complexity'] ... ['dst_is_top_venue', 'dst_is_english']
Dati pronti. Esegui ora le celle LGBM e XGBoost.

🧹 Memory cleanup: freed 18 objects: train_df, val_df, test_df, all_df, node_feats_full, node_feats_train, no

# MODEL 1 --> pair (one paper cite another)

In [8]:
pip install lightgbm

Python(55302) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


  Using cached lightgbm-4.6.0-py3-none-macosx_12_0_arm64.whl.metadata (17 kB)
Using cached lightgbm-4.6.0-py3-none-macosx_12_0_arm64.whl (1.6 MB)

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
import gc
import os
import lightgbm as lgb
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
)

assert 'pair_shapes' in dir(), "Esegui prima la cella di costruzione dei dati (sezioni 0-4)!"

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║            LIGHTGBM — LINK PREDICTION (MEMORY-SAFE)                    ║
# ║                                                                        ║
# ║  Con 63M+ coppie e 8 GB di RAM, serve:                                ║
# ║  1. Costruire i Dataset LightGBM manualmente con free_raw_data=True    ║
# ║  2. NON includere train nell'eval_set (evita duplicazione Dataset)     ║
# ║  3. Predire il test in blocchi per evitare picchi di RAM               ║
# ╚══════════════════════════════════════════════════════════════════════════╝

PAIRS_DIR = "model_splits/pairs"

# ── 1. Costruisci il train Dataset (LightGBM binna e poi libera i dati raw) ──
print("Building LightGBM train Dataset (binning 63M+ rows — may take a few minutes)...")
gc.collect()

X_train_mm = np.memmap(f"{PAIRS_DIR}/X_train.dat", dtype="float32", mode="r",
                       shape=tuple(pair_shapes["train"]))
y_train_mm = np.memmap(f"{PAIRS_DIR}/y_train.dat", dtype="int32", mode="r",
                       shape=(pair_shapes["train"][0],))

train_dataset = lgb.Dataset(
    X_train_mm, label=y_train_mm,
    feature_name=feature_names_xgb,
    free_raw_data=True,          # libera il riferimento ai dati raw dopo il binning
)
train_dataset.construct()        # forza il binning ora
del X_train_mm, y_train_mm      # rilascia il memmap
gc.collect()
print("  ✓ Train Dataset constructed and raw data freed")

# ── 2. Costruisci il val Dataset ──
print("Building LightGBM val Dataset...")
X_val_mm = np.memmap(f"{PAIRS_DIR}/X_val.dat", dtype="float32", mode="r",
                     shape=tuple(pair_shapes["val"]))
y_val_mm = np.memmap(f"{PAIRS_DIR}/y_val.dat", dtype="int32", mode="r",
                     shape=(pair_shapes["val"][0],))

val_dataset = lgb.Dataset(
    X_val_mm, label=y_val_mm,
    reference=train_dataset,     # usa le stesse bin boundaries del train
    free_raw_data=True,
)
val_dataset.construct()
del X_val_mm, y_val_mm
gc.collect()
print("  ✓ Val Dataset constructed and raw data freed")

# ── 3. Training ──
print("\nTraining LightGBM...")
scale_pos_weight_lgbm = train_dataset.get_label()
scale_pos_weight_lgbm = (scale_pos_weight_lgbm == 0).sum() / max((scale_pos_weight_lgbm == 1).sum(), 1)

params = {
    "objective": "binary",
    "metric": "binary_logloss",
    "boosting_type": "gbdt",
    "n_estimators": 1000,
    "max_depth": 6,
    "num_leaves": 63,
    "learning_rate": 0.05,
    "subsample": 0.9,
    "subsample_freq": 1,
    "colsample_bytree": 0.9,
    "min_child_samples": 20,
    "reg_lambda": 1.0,
    "scale_pos_weight": scale_pos_weight_lgbm,
    "random_state": 42,
    "n_jobs": -1,
    "verbose": -1,
}

lgbm_evals_result = {}

t0 = time.time()
lgbm_booster = lgb.train(
    params,
    train_dataset,
    num_boost_round=1000,
    valid_sets=[val_dataset],           # SOLO val — non duplicare il train
    valid_names=["val"],
    callbacks=[
        lgb.early_stopping(stopping_rounds=20, verbose=True),
        lgb.log_evaluation(period=50),
        lgb.record_evaluation(lgbm_evals_result),
    ],
)
lgbm_train_time = time.time() - t0
print(f"\nTraining time: {lgbm_train_time:.1f}s")
print(f"Best iteration: {lgbm_booster.best_iteration}")

# Libera i Dataset (non servono più)
del train_dataset, val_dataset
gc.collect()

# ── 4. Evaluation sul test set (in blocchi per non esplodere) ──
print("\nEvaluating on test set (chunked prediction)...")
PRED_CHUNK = 1_000_000

X_test_mm = np.memmap(f"{PAIRS_DIR}/X_test.dat", dtype="float32", mode="r",
                      shape=tuple(pair_shapes["test"]))
y_test_xgb = np.memmap(f"{PAIRS_DIR}/y_test.dat", dtype="int32", mode="r",
                       shape=(pair_shapes["test"][0],))

n_test = pair_shapes["test"][0]
lgbm_probs = np.empty(n_test, dtype=np.float64)

for start in range(0, n_test, PRED_CHUNK):
    end = min(start + PRED_CHUNK, n_test)
    chunk = np.array(X_test_mm[start:end])   # copia in RAM solo il blocco
    lgbm_probs[start:end] = lgbm_booster.predict(chunk)
    del chunk

lgbm_preds = (lgbm_probs >= 0.5).astype(int)
y_test_np = np.array(y_test_xgb)   # copia per sklearn

del X_test_mm, y_test_xgb
gc.collect()

print("\n--- LightGBM Link Prediction (Test Split) ---")
print(f"ROC-AUC : {roc_auc_score(y_test_np, lgbm_probs):.4f}")
print(f"PR-AUC  : {average_precision_score(y_test_np, lgbm_probs):.4f}")
print(f"Brier   : {brier_score_loss(y_test_np, lgbm_probs):.4f}")
print("\nClassification report:")
print(classification_report(y_test_np, lgbm_preds, digits=4))


# ── 5. Plots ──
fig, axes = plt.subplots(1, 3, figsize=(22, 6))

val_loss = lgbm_evals_result["val"]["binary_logloss"]
axes[0].plot(val_loss, label="Val logloss", color="tomato", linewidth=1.5)
axes[0].axvline(lgbm_booster.best_iteration - 1, color="gray", linestyle="--",
                label=f"Best iter ({lgbm_booster.best_iteration})")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Log Loss")
axes[0].set_title("LightGBM — Validation Loss Curve")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

cm_lgbm = confusion_matrix(y_test_np, lgbm_preds)
ConfusionMatrixDisplay(confusion_matrix=cm_lgbm).plot(cmap="Greens", ax=axes[1], colorbar=False)
axes[1].set_title("Confusion Matrix — LightGBM")

importance_vals = lgbm_booster.feature_importance(importance_type="gain")
lgbm_importance_df = pd.DataFrame({
    "feature": feature_names_xgb,
    "gain": importance_vals,
}).sort_values("gain", ascending=False).head(15)

axes[2].barh(lgbm_importance_df["feature"][::-1], lgbm_importance_df["gain"][::-1], color="mediumseagreen")
axes[2].set_title("Top 15 Feature Importances (Gain) — LightGBM")
axes[2].set_xlabel("Relative Importance")

plt.tight_layout()
plt.show()
display(lgbm_importance_df.head(15))


Building LightGBM train Dataset (binning 63M+ rows — may take a few minutes)...


: 

# MODEL 2 -> XGBOOST

In [ ]:
# # ORIGINALE
# import numpy as np
# import torch
# from xgboost import XGBClassifier
# from torch_geometric.utils import negative_sampling
# from sklearn.metrics import (
#     roc_auc_score, average_precision_score, brier_score_loss,
#     classification_report, confusion_matrix, ConfusionMatrixDisplay,
# )

# # --- 1. Build node feature matrix (same features as the GCN) ---
# node_feats = node_df[num_cols].to_numpy(dtype=np.float32)
# node_feats = np.nan_to_num(node_feats, nan=0.0, posinf=0.0, neginf=0.0)

# # --- 2. Negative sampling for each split (mirrors the GCN protocol) ---
# torch.manual_seed(42)
# num_nodes = len(node_df)

# neg_train = negative_sampling(
#     edge_index=train_pos,
#     num_nodes=num_nodes,
#     num_neg_samples=train_pos.size(1),
# )
# neg_test = negative_sampling(
#     edge_index=torch.cat([train_pos, val_pos], dim=1),
#     num_nodes=num_nodes,
#     num_neg_samples=test_pos.size(1),
# )

# # --- 3. Turn an edge_index tensor into a pair feature matrix (src || dst) ---
# def pairs_to_matrix(edge_index):
#     src = edge_index[0].cpu().numpy()
#     dst = edge_index[1].cpu().numpy()
#     return np.hstack([node_feats[src], node_feats[dst]])

# X_train_xgb = np.vstack([pairs_to_matrix(train_pos), pairs_to_matrix(neg_train)])
# y_train_xgb = np.concatenate([
#     np.ones(train_pos.size(1), dtype=np.int32),
#     np.zeros(neg_train.size(1), dtype=np.int32),
# ])

# X_test_xgb = np.vstack([pairs_to_matrix(test_pos), pairs_to_matrix(neg_test)])
# y_test_xgb = np.concatenate([
#     np.ones(test_pos.size(1), dtype=np.int32),
#     np.zeros(neg_test.size(1), dtype=np.int32),
# ])

# feature_names_xgb = [f"src_{c}" for c in num_cols] + [f"dst_{c}" for c in num_cols]

# print(f"Train pairs: {len(y_train_xgb):,} | Test pairs: {len(y_test_xgb):,}")

# # --- 4. Train XGBoost ---
# scale_pos_weight = (y_train_xgb == 0).sum() / max((y_train_xgb == 1).sum(), 1)

# xgb_model = XGBClassifier(
#     n_estimators=500,
#     max_depth=6,
#     learning_rate=0.05,
#     subsample=0.9,
#     colsample_bytree=0.9,
#     min_child_weight=2,
#     reg_lambda=1.0,
#     objective="binary:logistic",
#     eval_metric="logloss",
#     scale_pos_weight=scale_pos_weight,
#     tree_method="hist",
#     random_state=42,
#     n_jobs=-1,
# )
# xgb_model.fit(X_train_xgb, y_train_xgb, verbose=False)

# # --- 5. Evaluate on the held-out test edges ---
# xgb_probs = xgb_model.predict_proba(X_test_xgb)[:, 1]
# xgb_preds = (xgb_probs >= 0.5).astype(int)

# print("\nXGBoost link prediction — test split")
# print(f"ROC-AUC:  {roc_auc_score(y_test_xgb, xgb_probs):.4f}")
# print(f"PR-AUC:   {average_precision_score(y_test_xgb, xgb_probs):.4f}")
# print(f"Brier:    {brier_score_loss(y_test_xgb, xgb_probs):.4f}")
# print("\nClassification report")
# print(classification_report(y_test_xgb, xgb_preds, digits=4))

# cm_xgb = confusion_matrix(y_test_xgb, xgb_preds)
# ConfusionMatrixDisplay(confusion_matrix=cm_xgb).plot(cmap="Blues")
# plt.title("XGBoost link prediction: Confusion Matrix")
# plt.tight_layout()
# plt.show()

# # --- 6. Feature importance (gain-based) ---
# importance_df = pd.DataFrame(
#     {"feature": feature_names_xgb, "gain": xgb_model.feature_importances_}
# ).sort_values("gain", ascending=False)
# display(importance_df.head(20))

In [ ]:
# # HA LEAKAGE

# import polars as pl
# import numpy as np
# import torch
# import pandas as pd
# import matplotlib.pyplot as plt
# from xgboost import XGBClassifier
# from torch_geometric.utils import negative_sampling
# from sklearn.metrics import (
#     roc_auc_score, average_precision_score, brier_score_loss,
#     classification_report, confusion_matrix, ConfusionMatrixDisplay,
# )

# # ╔══════════════════════════════════════════════════════════════════════════╗
# # ║                        SUBSAMPLING CONFIGURATION                         ║
# # ╚══════════════════════════════════════════════════════════════════════════╝
# # Subsample percentages to save RAM and speed up training
# FRACTION_TRAIN = 0.05  
# FRACTION_VAL   = 0.10  
# FRACTION_TEST  = 0.10  

# # --- 0. Load Data ---
# print("Loading Parquet splits...")
# train_df = pl.read_parquet('model_splits/train.parquet')
# val_df = pl.read_parquet('model_splits/val.parquet')
# test_df = pl.read_parquet('model_splits/test.parquet')

# # Combine to build the global feature matrix and ID mapping
# all_df = pl.concat([train_df, val_df, test_df])

# # IMPORTANT: Ensure absolute chronological sorting
# all_df = all_df.sort("node_idx")

# # Get maximum node indices to strictly prevent temporal leakage during negative sampling
# num_train_nodes = train_df["node_idx"].max() + 1
# num_val_nodes   = max(train_df["node_idx"].max(), val_df["node_idx"].max()) + 1
# num_total_nodes = all_df["node_idx"].max() + 1

# id_to_idx = all_df.select(["id", "node_idx"])

# # --- 1. Build node feature matrix ---
# exclude_cols = ["id", "node_idx", "split", "year", "references", "target_variable"]
# feature_cols = [c for c in all_df.columns if c not in exclude_cols]

# print(f"Building feature matrix for {num_total_nodes:,} nodes using {len(feature_cols)} features...")
# node_feats = all_df.select(feature_cols).to_numpy()
# node_feats = np.nan_to_num(node_feats, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

# # Function to extract positive edges
# def get_edge_index(split_df):
#     edges = (
#         split_df.select(["node_idx", "references"])
#         .explode("references")
#         .drop_nulls()
#         .rename({"node_idx": "src", "references": "target_id"})
#     )
#     edges = edges.join(id_to_idx, left_on="target_id", right_on="id", how="inner")
#     edges = edges.rename({"node_idx": "dst"})
    
#     src_np = edges["src"].to_numpy()
#     dst_np = edges["dst"].to_numpy()
#     return torch.tensor(np.vstack([src_np, dst_np]), dtype=torch.long)

# # Function to randomly subsample edges
# def subsample_edges(edge_index, fraction):
#     if fraction >= 1.0: return edge_index
#     num_edges = edge_index.size(1)
#     num_samples = max(1, int(num_edges * fraction))
#     idx = torch.randperm(num_edges)[:num_samples]
#     return edge_index[:, idx]

# print("Extracting positive edges...")
# train_pos_full = get_edge_index(train_df)
# val_pos_full   = get_edge_index(val_df)
# test_pos_full  = get_edge_index(test_df)

# print("Applying subsampling...")
# torch.manual_seed(42) # Reproducibility
# train_pos = subsample_edges(train_pos_full, FRACTION_TRAIN)
# val_pos   = subsample_edges(val_pos_full, FRACTION_VAL)
# test_pos  = subsample_edges(test_pos_full, FRACTION_TEST)

# print(f"  - Train edges : {train_pos.size(1):,} (out of {train_pos_full.size(1):,})")
# print(f"  - Val edges   : {val_pos.size(1):,} (out of {val_pos_full.size(1):,})")
# print(f"  - Test edges  : {test_pos.size(1):,} (out of {test_pos_full.size(1):,})")

# # --- 2. Temporal Negative Sampling ---
# print("\nGenerating negative samples (strictly respecting timelines)...")

# # TRAIN negatives: Only nodes that existed during training time
# neg_train = negative_sampling(
#     edge_index=train_pos,
#     num_nodes=num_train_nodes,
#     num_neg_samples=train_pos.size(1),
# )

# # VAL negatives: Nodes up to validation time
# neg_val = negative_sampling(
#     edge_index=torch.cat([train_pos, val_pos], dim=1),
#     num_nodes=num_val_nodes,
#     num_neg_samples=val_pos.size(1),
# )

# # TEST negatives: Can sample from any available node
# neg_test = negative_sampling(
#     edge_index=torch.cat([train_pos, val_pos, test_pos], dim=1),
#     num_nodes=num_total_nodes,
#     num_neg_samples=test_pos.size(1),
# )

# # --- 3. Build Matrices for XGBoost ---
# def pairs_to_matrix(edge_index):
#     src = edge_index[0].cpu().numpy()
#     dst = edge_index[1].cpu().numpy()
#     return np.hstack([node_feats[src], node_feats[dst]])

# print("\nCreating tabular datasets...")
# # TRAIN
# X_train_xgb = np.vstack([pairs_to_matrix(train_pos), pairs_to_matrix(neg_train)])
# y_train_xgb = np.concatenate([np.ones(train_pos.size(1), dtype=np.int32), 
#                               np.zeros(neg_train.size(1), dtype=np.int32)])

# # VAL
# X_val_xgb = np.vstack([pairs_to_matrix(val_pos), pairs_to_matrix(neg_val)])
# y_val_xgb = np.concatenate([np.ones(val_pos.size(1), dtype=np.int32), 
#                             np.zeros(neg_val.size(1), dtype=np.int32)])

# # TEST
# X_test_xgb = np.vstack([pairs_to_matrix(test_pos), pairs_to_matrix(neg_test)])
# y_test_xgb = np.concatenate([np.ones(test_pos.size(1), dtype=np.int32), 
#                              np.zeros(neg_test.size(1), dtype=np.int32)])

# feature_names_xgb = [f"src_{c}" for c in feature_cols] + [f"dst_{c}" for c in feature_cols]

# # --- 4. Train XGBoost ---
# print("\nTraining XGBoost with Early Stopping on Validation Set...")
# scale_pos_weight = (y_train_xgb == 0).sum() / max((y_train_xgb == 1).sum(), 1)

# xgb_model = XGBClassifier(
#     n_estimators=1000,          # Increased, early stopping will halt it
#     max_depth=6,
#     learning_rate=0.05,
#     subsample=0.9,
#     colsample_bytree=0.9,
#     min_child_weight=2,
#     reg_lambda=1.0,
#     objective="binary:logistic",
#     eval_metric="logloss",
#     scale_pos_weight=scale_pos_weight,
#     tree_method="hist",
#     early_stopping_rounds=20,   # Stops if val loss doesn't improve for 20 rounds
#     random_state=42,
#     n_jobs=-1,
# )

# # We now pass the validation set so XGBoost knows when to stop!
# xgb_model.fit(
#     X_train_xgb, y_train_xgb,
#     eval_set=[(X_val_xgb, y_val_xgb)],
#     verbose=False
# )
# print(f"Training stopped at iteration {xgb_model.best_iteration}")

# # --- 5. Evaluation ---
# print("\nEvaluating on the Test Set...")
# xgb_probs = xgb_model.predict_proba(X_test_xgb)[:, 1]
# xgb_preds = (xgb_probs >= 0.5).astype(int)

# print("\n--- XGBoost Link Prediction (Test Split) ---")
# print(f"ROC-AUC : {roc_auc_score(y_test_xgb, xgb_probs):.4f}")
# print(f"PR-AUC  : {average_precision_score(y_test_xgb, xgb_probs):.4f}")
# print(f"Brier   : {brier_score_loss(y_test_xgb, xgb_probs):.4f}")
# print("\nClassification report:")
# print(classification_report(y_test_xgb, xgb_preds, digits=4))

# # Confusion Matrix & Feature Importance Plot
# fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# # Subplot 1: Confusion Matrix
# cm_xgb = confusion_matrix(y_test_xgb, xgb_preds)
# ConfusionMatrixDisplay(confusion_matrix=cm_xgb).plot(cmap="Blues", ax=axes[0], colorbar=False)
# axes[0].set_title("Confusion Matrix")

# # Subplot 2: Feature Importance
# importance_df = pd.DataFrame(
#     {"feature": feature_names_xgb, "gain": xgb_model.feature_importances_}
# ).sort_values("gain", ascending=False).head(15)

# axes[1].barh(importance_df["feature"][::-1], importance_df["gain"][::-1], color="steelblue")
# axes[1].set_title("Top 15 Feature Importances (Gain)")
# axes[1].set_xlabel("Relative Importance")

# plt.tight_layout()
# plt.show()

# display(importance_df.head(15))

In [ ]:
import gc
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
)

assert 'pair_shapes' in dir(), "Esegui prima la cella di costruzione dei dati (sezioni 0-4)!"

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║             XGBOOST — LINK PREDICTION (MEMORY-SAFE)                    ║
# ║                                                                        ║
# ║  Stesse ottimizzazioni di LightGBM:                                   ║
# ║  - Solo val nell'eval_set (evita duplicazione dei dati)                ║
# ║  - Predizione test in blocchi                                          ║
# ╚══════════════════════════════════════════════════════════════════════════╝

PAIRS_DIR = "model_splits/pairs"

gc.collect()

# ── 1. Carica dati via memmap ──
print("Loading pair data via memmap...")
X_train_mm = np.memmap(f"{PAIRS_DIR}/X_train.dat", dtype="float32", mode="r",
                       shape=tuple(pair_shapes["train"]))
y_train_mm = np.memmap(f"{PAIRS_DIR}/y_train.dat", dtype="int32", mode="r",
                       shape=(pair_shapes["train"][0],))
X_val_mm = np.memmap(f"{PAIRS_DIR}/X_val.dat", dtype="float32", mode="r",
                     shape=tuple(pair_shapes["val"]))
y_val_mm = np.memmap(f"{PAIRS_DIR}/y_val.dat", dtype="int32", mode="r",
                     shape=(pair_shapes["val"][0],))

scale_pos_weight = (y_train_mm == 0).sum() / max((y_train_mm == 1).sum(), 1)
print(f"  scale_pos_weight: {scale_pos_weight:.3f}")

# ── 2. Training ──
print("\nTraining XGBoost with Early Stopping...")
xgb_model = XGBClassifier(
    n_estimators=1000,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    min_child_weight=2,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
    tree_method="hist",
    early_stopping_rounds=20,
    random_state=42,
    n_jobs=-1,
)

t0 = time.time()
xgb_model.fit(
    X_train_mm, y_train_mm,
    eval_set=[(X_val_mm, y_val_mm)],    # SOLO val — non duplicare il train
    verbose=50,
)
xgb_train_time = time.time() - t0
xgb_evals_result = xgb_model.evals_result()

print(f"\nTraining stopped at iteration : {xgb_model.best_iteration}")
print(f"Training time                 : {xgb_train_time:.1f}s")

del X_train_mm, y_train_mm, X_val_mm, y_val_mm
gc.collect()

# ── 3. Evaluation in blocchi ──
print("\nEvaluating on test set (chunked prediction)...")
PRED_CHUNK = 1_000_000

X_test_mm = np.memmap(f"{PAIRS_DIR}/X_test.dat", dtype="float32", mode="r",
                      shape=tuple(pair_shapes["test"]))
y_test_mm = np.memmap(f"{PAIRS_DIR}/y_test.dat", dtype="int32", mode="r",
                      shape=(pair_shapes["test"][0],))

n_test = pair_shapes["test"][0]
xgb_probs = np.empty(n_test, dtype=np.float64)

for start in range(0, n_test, PRED_CHUNK):
    end = min(start + PRED_CHUNK, n_test)
    chunk = np.array(X_test_mm[start:end])
    xgb_probs[start:end] = xgb_model.predict_proba(chunk)[:, 1]
    del chunk

xgb_preds = (xgb_probs >= 0.5).astype(int)

# Riusa y_test_np dalla cella LightGBM se disponibile, altrimenti carica
if 'y_test_np' not in dir():
    y_test_np = np.array(y_test_mm)
del X_test_mm, y_test_mm
gc.collect()

print("\n--- XGBoost Link Prediction (Test Split) ---")
print(f"ROC-AUC : {roc_auc_score(y_test_np, xgb_probs):.4f}")
print(f"PR-AUC  : {average_precision_score(y_test_np, xgb_probs):.4f}")
print(f"Brier   : {brier_score_loss(y_test_np, xgb_probs):.4f}")
print("\nClassification report:")
print(classification_report(y_test_np, xgb_preds, digits=4))


# ── 4. Plots ──
fig, axes = plt.subplots(1, 3, figsize=(22, 6))

xgb_val_loss = xgb_evals_result["validation_0"]["logloss"]
axes[0].plot(xgb_val_loss, label="Val logloss", color="tomato", linewidth=1.5)
axes[0].axvline(xgb_model.best_iteration, color="gray", linestyle="--",
                label=f"Best iter ({xgb_model.best_iteration})")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Log Loss")
axes[0].set_title("XGBoost — Validation Loss Curve")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

cm_xgb = confusion_matrix(y_test_np, xgb_preds)
ConfusionMatrixDisplay(confusion_matrix=cm_xgb).plot(cmap="Blues", ax=axes[1], colorbar=False)
axes[1].set_title("Confusion Matrix — XGBoost")

importance_df = pd.DataFrame(
    {"feature": feature_names_xgb, "gain": xgb_model.feature_importances_}
).sort_values("gain", ascending=False).head(15)

axes[2].barh(importance_df["feature"][::-1], importance_df["gain"][::-1], color="steelblue")
axes[2].set_title("Top 15 Feature Importances (Gain) — XGBoost")
axes[2].set_xlabel("Relative Importance")

plt.tight_layout()
plt.show()
display(importance_df.head(15))


# ╔══════════════════════════════════════════════════════════════════════════╗
# ║                    FAIR COMPARISON SUMMARY                             ║
# ╚══════════════════════════════════════════════════════════════════════════╝
print("\n══════════════ FAIR COMPARISON ══════════════")
print(f"{'Metric':<14} {'LightGBM':>10} {'XGBoost':>10}")
print(f"{'─'*36}")
print(f"{'ROC-AUC':<14} {roc_auc_score(y_test_np, lgbm_probs):>10.4f} {roc_auc_score(y_test_np, xgb_probs):>10.4f}")
print(f"{'PR-AUC':<14} {average_precision_score(y_test_np, lgbm_probs):>10.4f} {average_precision_score(y_test_np, xgb_probs):>10.4f}")
print(f"{'Brier':<14} {brier_score_loss(y_test_np, lgbm_probs):>10.4f} {brier_score_loss(y_test_np, xgb_probs):>10.4f}")
print(f"{'Train time':<14} {lgbm_train_time:>9.1f}s {xgb_train_time:>9.1f}s")
print(f"{'Best iter':<14} {lgbm_booster.best_iteration:>10} {xgb_model.best_iteration:>10}")


In [ ]:
df_check = pd.read_parquet(
    "papers_with_features.parquet",
    columns=["n_references", "target_variable"]
)

print(df_check.head(10))
print(f"\nCorrelazione n_references vs target_variable: "
      f"{df_check['n_references'].corr(df_check['target_variable']):.4f}")
print(f"Sono identici: "
      f"{(df_check['n_references'] == df_check['target_variable']).mean():.1%} delle righe")
print(f"\nn_references - sample: {df_check['n_references'].head(5).tolist()}")
print(f"target_variable - sample: {df_check['target_variable'].head(5).tolist()}")

# MODEL 3 -> Random Forest, using pair-level features

## Pair-Level Baseline for Citation Link Prediction

The GCN section models the citation graph directly. For this target, the supervised example is a **paper pair**: source paper $A$ and candidate target paper $B$, with label $y=1$ if $A$ cites $B$ and $y=0$ otherwise.

That means the right tabular baseline should:
- build one row per paper pair,
- use only metadata available before the link decision,
- avoid citation-count leakage such as `n_citation` or `impact_score`,
- stay interpretable so the team can explain the result with SHAP and uncertainty.

Below, the notebook builds pair-level features from what is already in the dataset: authors, keywords, titles, abstracts, venue, year, and reference lists.

In [ ]:
# %pip install -q scikit-learn shap

In [ ]:
import re
import random
from bisect import bisect_right

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False

random.seed(42)
np.random.seed(42)


In [ ]:
def as_list(value):
    if value is None:
        return []
    if isinstance(value, float) and np.isnan(value):
        return []
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    return [value]


def normalize_scalar(value):
    if value is None:
        return ""
    if isinstance(value, float) and np.isnan(value):
        return ""
    text = str(value).strip().lower()
    return re.sub(r"\s+", " ", text)


def extract_author_names(authors):
    names = set()
    for item in as_list(authors):
        if isinstance(item, dict):
            name = item.get("name")
        elif hasattr(item, "get"):
            name = item.get("name")
        else:
            name = item
        name = normalize_scalar(name)
        if name:
            names.add(name)
    return names


def extract_strings(values):
    items = set()
    for item in as_list(values):
        token = normalize_scalar(item)
        if token:
            items.add(token)
    return items


def safe_int(value):
    if value is None:
        return 0
    if isinstance(value, float) and np.isnan(value):
        return 0
    return int(value)


def tokenize_text(text):
    text = normalize_scalar(text)
    if not text:
        return set()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return {tok for tok in text.split() if tok}


def jaccard(left, right):
    union = left | right
    if not union:
        return 0.0
    return len(left & right) / len(union)


Up to this point, our Exploratory Data Analysis (EDA) focused on **Node-Level Features**—metrics describing individual papers (e.g., `impact_score`, `citation_per_year`, `venue_tier`). While these are excellent for understanding global trends, our ultimate machine learning goal is **Link Prediction**: determining whether Paper A cites Paper B.

To predict links, we cannot look at papers in isolation. We must evaluate the *relationship* between them. 

If we try to predict a citation by feeding the model Paper B's total `impact_score` or `n_citation`, the model will suffer from **Target Leakage**. It would simply learn to predict that "highly cited papers get cited," effectively using future knowledge (citations accumulated up to 2024) to predict a past event. 

Instead, we force the model to rely purely on information available at the exact moment of publication. We do this by calculating **Pair-Level Features**:
1. **Jaccard Similarities:** We measure the mathematical overlap between the two papers' metadata (Authors, Keywords, References, Titles, and Abstracts). A higher Jaccard index implies stronger topical or social proximity.
2. **Temporal Features:** We calculate `year_gap` to understand the age difference between the citing and cited paper, and boolean flags like `same_decade` to capture chronological clustering.
3. **Venue & Overlap Flags:** Simple binary indicators (e.g., `same_venue`, `author_overlap_any`) to capture direct matches.

By constructing a balanced dataset of 10,000 pairs (50% actual citations, 50% random non-citations), we frame the link prediction task as a binary classification problem based purely on relational similarity.

In [ ]:
print("Building pair-level dataset from paper metadata and reference links...")

# Keep this in DuckDB: only the lightweight id/year index is loaded here.
# The full paper metadata for the sampled pairs will be fetched later,
# after the positive/negative pairs are known.
all_ids_years = con.sql("""
SELECT id, year
FROM papers
WHERE id IS NOT NULL AND TRIM(id) <> '' AND year IS NOT NULL
""").df()

print(f"Indexed {len(all_ids_years):,} paper ids with year metadata")

In [ ]:

positive_pairs = con.sql("""
SELECT DISTINCT
    p.id AS src_id,
    ref AS dst_id
FROM papers p
CROSS JOIN UNNEST(p."references") AS t(ref)
JOIN papers d ON d.id = ref
WHERE p.id IS NOT NULL
  AND d.id IS NOT NULL
  AND TRIM(p.id) <> ''
  AND TRIM(ref) <> ''
  AND p.id <> ref
  AND p.year IS NOT NULL
  AND d.year IS NOT NULL
""").df()

if positive_pairs.empty:
    raise RuntimeError("No citation edges were found in the dataset.")

In [ ]:
sample_n = min(5000, len(positive_pairs))
positive_pairs = positive_pairs.sample(n=sample_n, random_state=42).reset_index(drop=True)

# Build a full paper-id/year index for negative sampling.
all_id_to_year = dict(zip(all_ids_years["id"], all_ids_years["year"]))
all_years_sorted = sorted(all_ids_years["year"].dropna().astype(int).unique().tolist())
all_ids_by_year = {year: all_ids_years.loc[all_ids_years["year"] == year, "id"].tolist() for year in all_years_sorted}
all_prefix_ids = {}
running_ids = []
for year in all_years_sorted:
    running_ids.extend(all_ids_by_year[year])
    all_prefix_ids[year] = running_ids.copy()

# Only fetch the reference lists for the sampled source papers.
source_ids_df = pd.DataFrame({"id": positive_pairs["src_id"].unique()})
con.register("source_ids", source_ids_df)
source_refs_df = con.sql("""
SELECT p.id, p."references" AS refs
FROM papers p
JOIN source_ids s ON p.id = s.id
""").df()
source_ref_lookup = {row.id: extract_strings(row.refs) for row in source_refs_df.itertuples(index=False)}

# Alias names used by the following cell.
id_to_year = all_id_to_year
years_sorted = all_years_sorted
prefix_ids = all_prefix_ids
id_to_refs = source_ref_lookup

# Sample negatives from all papers that are not already cited by the source.
def sample_negative_target(src_id):
    src_year = id_to_year[src_id]
    idx = bisect_right(years_sorted, src_year) - 1
    if idx < 0:
        return None
    pool_year = years_sorted[idx]
    pool = prefix_ids[pool_year]
    banned = id_to_refs.get(src_id, set()) | {src_id}
    if len(pool) <= len(banned):
        return None
    for _ in range(50):
        candidate = random.choice(pool)
        if candidate not in banned and id_to_year.get(candidate, src_year) <= src_year:
            return candidate
    for candidate in random.sample(pool, min(len(pool), 200)):
        if candidate not in banned and id_to_year.get(candidate, src_year) <= src_year:
            return candidate
    return None

negative_rows = []
for src_id in positive_pairs["src_id"].tolist():
    dst_id = sample_negative_target(src_id)
    if dst_id is not None:
        negative_rows.append((src_id, dst_id))

negative_pairs = pd.DataFrame(negative_rows, columns=["src_id", "dst_id"])
if negative_pairs.empty:
    raise RuntimeError("Failed to sample negative pairs.")

positive_pairs = positive_pairs.iloc[: len(negative_pairs)].copy()
positive_pairs["label"] = 1
negative_pairs["label"] = 0
pair_df = pd.concat([positive_pairs, negative_pairs], ignore_index=True)
pair_df = pair_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Fetch only the paper metadata that actually appears in the sampled pairs.
pair_ids_df = pd.DataFrame({"id": pd.unique(pd.concat([pair_df["src_id"], pair_df["dst_id"]], ignore_index=True))})
con.register("pair_ids", pair_ids_df)
papers_df = con.sql("""
SELECT
    id,
    year,
    venue,
    title,
    abstract,
    authors,
    keywords,
    "references" AS refs,
    n_authors,
    n_keywords,
    n_references,
    title_len,
    abstract_len
FROM papers
WHERE id IN (SELECT id FROM pair_ids)
""").df()

paper_lookup = papers_df.set_index("id").to_dict(orient="index")

print(f"Sampled positive pairs: {len(positive_pairs):,}")
print(f"Sampled negative pairs: {len(negative_pairs):,}")
print(f"Fetched metadata for {len(papers_df):,} unique papers involved in the sampled pairs.")

In [ ]:


def sample_negative_target(src_id):
    src_year = id_to_year[src_id]
    idx = bisect_right(years_sorted, src_year) - 1
    if idx < 0:
        return None
    pool_year = years_sorted[idx]
    pool = prefix_ids[pool_year]
    banned = id_to_refs.get(src_id, set()) | {src_id}
    if len(pool) <= len(banned):
        return None
    for _ in range(50):
        candidate = random.choice(pool)
        if candidate not in banned and id_to_year.get(candidate, src_year) <= src_year:
            return candidate
    for candidate in random.sample(pool, min(len(pool), 200)):
        if candidate not in banned and id_to_year.get(candidate, src_year) <= src_year:
            return candidate
    return None

In [ ]:
print(f"Pair table ready: {len(pair_df):,} rows")
display(pair_df.head())

## The Tabular Supervised Baseline (Random Forest)

### The Objective
Before deploying a complex, graph-native approach (like a Graph Convolutional Network), it is best practice to establish a robust **Supervised Tabular Baseline**. We use a Random Forest Classifier for this task because it handles non-linear relationships well, is robust to outliers in our text similarities, and provides excellent interpretability without requiring feature scaling.

### Feature Selection
Notice the `feature_cols` list below. **We strictly exclude all individual quality metrics** (`impact_score`, `venue_tier`, `n_citation`). The model is only allowed to see:
* Jaccard similarities (Text, Authors, References)
* Temporal distances (`year_gap`, `same_year`)
* Absolute lengths (Number of authors, abstract length)

This ensures our model genuinely learns *why* citations happen based on content and context, rather than "cheating" by looking at global popularity metrics.

### Model Configuration & Evaluation Strategy
1. **Stratified Splitting:** We use an 80/20 train-test split, stratified on our target `label` to ensure perfectly balanced classes in both sets.
2. **Hyperparameters:** We configure the Random Forest with `n_estimators=300` for stability, `min_samples_leaf=2` to prevent overfitting on highly specific paper pairs, and `class_weight="balanced_subsample"` to handle bootstrap-level imbalances.
3. **Interpretability:** Beyond standard metrics (ROC-AUC, PR-AUC, F1-Score), we prioritize interpretability. We utilize **Permutation Importance** (and SHAP values, if available) to explicitly rank which pair-level signals drive citation decisions, and we calculate the **Brier Score** to ensure our predicted citation probabilities are well-calibrated.

In [ ]:


def build_pair_features(src_id, dst_id, label):
    src = paper_lookup[src_id]
    dst = paper_lookup[dst_id]

    src_authors = extract_author_names(src["authors"])
    dst_authors = extract_author_names(dst["authors"])
    src_keywords = extract_strings(src["keywords"])
    dst_keywords = extract_strings(dst["keywords"])
    src_refs = extract_strings(src["refs"])
    dst_refs = extract_strings(dst["refs"])
    src_title = tokenize_text(src["title"])
    dst_title = tokenize_text(dst["title"])
    src_abstract = tokenize_text(src["abstract"])
    dst_abstract = tokenize_text(dst["abstract"])

    src_venue = normalize_scalar(src["venue"])
    dst_venue = normalize_scalar(dst["venue"])
    src_year = int(src["year"])
    dst_year = int(dst["year"])

    author_overlap = len(src_authors & dst_authors)
    keyword_overlap = len(src_keywords & dst_keywords)
    ref_overlap = len(src_refs & dst_refs)

    return {
        "src_id": src_id,
        "dst_id": dst_id,
        "label": label,
        "year_gap": src_year - dst_year,
        "src_year": src_year,
        "dst_year": dst_year,
        "same_year": int(src_year == dst_year),
        "same_decade": int(src_year // 10 == dst_year // 10),
        "source_not_earlier": int(src_year >= dst_year),
        "same_venue": int(src_venue != "" and src_venue == dst_venue),
        "author_overlap": author_overlap,
        "author_overlap_any": int(author_overlap > 0),
        "author_jaccard": jaccard(src_authors, dst_authors),
        "keyword_overlap": keyword_overlap,
        "keyword_overlap_any": int(keyword_overlap > 0),
        "keyword_jaccard": jaccard(src_keywords, dst_keywords),
        "title_jaccard": jaccard(src_title, dst_title),
        "abstract_jaccard": jaccard(src_abstract, dst_abstract),
        "bibliographic_coupling": ref_overlap,
        "bibliographic_coupling_any": int(ref_overlap > 0),
        "reference_jaccard": jaccard(src_refs, dst_refs),
        "source_n_authors": safe_int(src["n_authors"]),
        "dst_n_authors": safe_int(dst["n_authors"]),
        "source_n_keywords": safe_int(src["n_keywords"]),
        "dst_n_keywords": safe_int(dst["n_keywords"]),
        "source_n_references": safe_int(src["n_references"]),
        "dst_n_references": safe_int(dst["n_references"]),
        "source_title_len": safe_int(src["title_len"]),
        "dst_title_len": safe_int(dst["title_len"]),
        "source_abstract_len": safe_int(src["abstract_len"]),
        "dst_abstract_len": safe_int(dst["abstract_len"]),
    }

In [ ]:
pair_feature_rows = [build_pair_features(row.src_id, row.dst_id, int(row.label)) for row in pair_df.itertuples(index=False)]
pair_features_df = pd.DataFrame(pair_feature_rows)

In [ ]:

print(f"Pair rows built: {len(pair_features_df):,}")
display(pair_features_df.head())


In [ ]:
feature_cols = [
    "year_gap",
    "src_year",
    "dst_year",
    "same_year",
    "same_decade",
    "source_not_earlier",
    "same_venue",
    "author_overlap",
    "author_overlap_any",
    "author_jaccard",
    "keyword_overlap",
    "keyword_overlap_any",
    "keyword_jaccard",
    "title_jaccard",
    "abstract_jaccard",
    "bibliographic_coupling",
    "bibliographic_coupling_any",
    "reference_jaccard",
    "source_n_authors",
    "dst_n_authors",
    "source_n_keywords",
    "dst_n_keywords",
    "source_n_references",
    "dst_n_references",
    "source_title_len",
    "dst_title_len",
    "source_abstract_len",
    "dst_abstract_len",
]

X = pair_features_df[feature_cols].fillna(0)
y = pair_features_df["label"].astype(int)
meta = pair_features_df[["src_id", "dst_id"]].copy()

X_train, X_test, y_train, y_test, meta_train, meta_test = train_test_split(
    X,
    y,
    meta,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

In [ ]:

model = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)

probs = model.predict_proba(X_test)[:, 1]
preds = (probs >= 0.5).astype(int)

print("\nEvaluation on held-out pairs")
print(f"ROC-AUC:  {roc_auc_score(y_test, probs):.4f}")
print(f"PR-AUC:   {average_precision_score(y_test, probs):.4f}")
print(f"Brier:    {brier_score_loss(y_test, probs):.4f}")
print("\nClassification report")
print(classification_report(y_test, preds, digits=4))

In [ ]:
cm = confusion_matrix(y_test, preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Blues")
plt.title("Pair-level Random Forest: Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:


tree_probs = np.vstack([tree.predict_proba(X_test)[:, 1] for tree in model.estimators_])
uncertainty_df = pd.DataFrame(
    {
        "probability": probs,
        "prob_std": tree_probs.std(axis=0),
        "entropy": -(probs * np.log2(np.clip(probs, 1e-9, 1.0)) + (1 - probs) * np.log2(np.clip(1 - probs, 1e-9, 1.0))),
        "y_true": y_test.to_numpy(),
    },
    index=X_test.index,
)

In [ ]:


print("\nUncertainty summary")
display(uncertainty_df.describe().T)
print("\nMost uncertain test predictions")
display(
    pd.concat([meta_test, uncertainty_df], axis=1)
      .sort_values("prob_std", ascending=False)
      .head(10)
)

In [ ]:

print("\nFeature importance (permutation)")
perm = permutation_importance(
    model,
    X_test,
    y_test,
    n_repeats=5,
    random_state=42,
    scoring="average_precision",
)
importance_df = pd.DataFrame(
    {
        "feature": feature_cols,
        "importance": perm.importances_mean,
        "std": perm.importances_std,
    }
).sort_values("importance", ascending=False)
display(importance_df.head(15))

In [ ]:
if HAS_SHAP:
    print("\nSHAP summary plot")
    shap_sample = X_test.sample(n=min(1000, len(X_test)), random_state=42)
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(shap_sample)
    shap.summary_plot(
        shap_values[1] if isinstance(shap_values, list) else shap_values,
        shap_sample,
        feature_names=feature_cols,
        show=True,
    )
else:
    print("\nSHAP is not available in the current environment; permutation importance is shown instead.")

### Why this baseline is a better fit

This section is aligned with the target because each example is explicitly a paper pair. It is also easier to explain:
- SHAP highlights which pair-level signals drive a citation decision,
- uncertainty from the forest shows which predictions are ambiguous,
- the features are interpretable and do not rely on citation-count leakage.

So the narrative is: the GCN is a graph-native approach, but the tabular pair model is the clean supervised baseline for the actual link-classification task.

In [ ]:
# FORSE DA TOGLIERE 

# import networkx as nx
# import matplotlib.pyplot as plt
# import numpy as np

# SEED = 42
# N_NODES = 100

# # --- 1. Pick the 100 most-connected papers (in + out degree combined) ---
# edges_arr = edge_df[['src_idx', 'dst_idx']].to_numpy()

# # Count appearances on either end
# all_endpoints = np.concatenate([edges_arr[:, 0], edges_arr[:, 1]])
# unique, counts = np.unique(all_endpoints, return_counts=True)
# order = np.argsort(-counts)  # descending

# selected = set(unique[order[:N_NODES]].tolist())

# # Keep edges whose BOTH endpoints are in the sample
# mask = np.isin(edges_arr[:, 0], list(selected)) & np.isin(edges_arr[:, 1], list(selected))
# sub_edges = edges_arr[mask]

# # --- 2. Build directed graph ---
# G = nx.DiGraph()
# G.add_nodes_from(selected)
# G.add_edges_from(sub_edges.tolist())

# # Drop isolated nodes (no edges to other sampled nodes) so the plot stays clean
# G.remove_nodes_from([n for n in list(G.nodes()) if G.degree(n) == 0])

# print(f"Subgraph: {G.number_of_nodes()} nodes, {G.number_of_edges()} citation links")

# # --- 3. Visual encoding ---
# nodes_list = list(G.nodes())
# if 'year' in node_df.columns:
#     node_years = node_df.set_index('node_idx').loc[nodes_list, 'year'].to_numpy()
# else:
#     node_years = None

# in_deg = np.array([G.in_degree(n) for n in nodes_list])
# node_sizes = 60 + in_deg * 25

# # --- 4. Plot ---
# fig, ax = plt.subplots(figsize=(14, 11))
# pos = nx.spring_layout(G, seed=SEED, k=1.2 / np.sqrt(max(G.number_of_nodes(), 1)), iterations=150)

# nx.draw_networkx_edges(
#     G, pos, ax=ax,
#     edge_color='#999', alpha=0.35, width=0.6,
#     arrows=True, arrowsize=7, arrowstyle='->',
#     connectionstyle='arc3,rad=0.1',
# )

# nodes_drawn = nx.draw_networkx_nodes(
#     G, pos, ax=ax,
#     nodelist=nodes_list,
#     node_size=node_sizes,
#     node_color=node_years if node_years is not None else '#1f77b4',
#     cmap='viridis',
#     edgecolors='white', linewidths=0.8,
#     alpha=0.95,
# )

# # Label only the top 10 most-cited papers
# top_nodes = sorted(nodes_list, key=lambda n: G.in_degree(n), reverse=True)[:10]
# nx.draw_networkx_labels(
#     G, pos, ax=ax,
#     labels={n: str(n) for n in top_nodes},
#     font_size=8,
# )

# if node_years is not None:
#     cbar = plt.colorbar(nodes_drawn, ax=ax, shrink=0.7, pad=0.01)
#     cbar.set_label('Publication year')

# ax.set_title(
#     f"Citation graph — {G.number_of_nodes()} papers, {G.number_of_edges()} links "
#     f"(top-{N_NODES} most-connected)",
#     fontsize=13,
# )
# ax.set_axis_off()
# plt.tight_layout()
# plt.show()